# ML Trading System - XGBoost Version

## STEP 1: Import Libraries & Connect MT5

In [1]:
import numpy as np
import pandas as pd
import MetaTrader5 as mt5
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import onnxruntime as ort
from datetime import datetime, timedelta
import warnings
import json
import os
import time
import shutil
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

print("XGBoost version:", xgb.__version__)
print("NumPy version:", np.__version__)

# Initialize MT5 (tanpa login - koneksi langsung)
if not mt5.initialize():
    print(f"MT5 initialization failed: {mt5.last_error()}")
else:
    print(f"MT5 connected successfully!")
    print(f"Terminal: {mt5.terminal_info().name}")
    print(f"Account: {mt5.account_info().login}")
    print(f"Server: {mt5.account_info().server}")

XGBoost version: 3.2.0
NumPy version: 2.4.3
MT5 connected successfully!
Terminal: MetaTrader 5
Account: 463109478
Server: Exness-MT5Trial17


## STEP 2: Download Data dari MT5 (Valetax)
Mengambil data OHLCV dari MT5 untuk training. Kita ambil data M15 (15 menit) agar cukup detail untuk S/R detection.

In [3]:
# ============================================================
# CONFIGURATION - Ganti SYMBOL saja, semua otomatis!
# ============================================================
SYMBOL = "XAUUSD.vxc"        # <<< GANTI PAIR DI SINI
TIMEFRAME = mt5.TIMEFRAME_M15
TIMEFRAME_NAME = "M15"
NUM_BARS = 95000             # Jumlah bar data untuk training
LOOKBACK = 60               # 60 candles lookback window
FUTURE_BARS = 4              # Prediksi 4 bar ke depan

# Pairs yang tersedia di Valetax:
# "EURUSD.vxc", "GBPUSD.vxc", "USDJPY.vxc", "BTCUSD.vxc", "XAUUSD.vxc"
# "GBPJPY.vxc", "AUDUSD.vxc", "NZDUSD.vxc", "USDCAD.vxc", "USDCHF.vxc"

# ============================================================
# DOWNLOAD DATA (M15 + H1 for multi-timeframe context)
# ============================================================
def download_mt5_data(symbol, timeframe, num_bars):
    """Download OHLCV data from MT5 - works for any pair"""
    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        print(f"Symbol {symbol} tidak ditemukan!")
        symbols = mt5.symbols_get()
        all_symbols = [s.name for s in symbols if '.vxc' in s.name]
        print(f"Available .vxc symbols: {all_symbols[:20]}")
        return None, None
    
    if not symbol_info.visible:
        mt5.symbol_select(symbol, True)
    
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, num_bars)
    if rates is None or len(rates) == 0:
        print(f"Failed to get data: {mt5.last_error()}")
        return None, None
    
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    df.rename(columns={'tick_volume': 'volume'}, inplace=True)
    
    print(f"Symbol: {symbol}")
    print(f"Data downloaded: {len(df)} bars")
    print(f"Period: {df.index[0]} to {df.index[-1]}")
    print(f"Point: {symbol_info.point}, Digits: {symbol_info.digits}")
    print(f"Spread: {symbol_info.spread} points = {symbol_info.spread * symbol_info.point:.5f} price")
    print(f"Tick Value: {symbol_info.trade_tick_value}, Tick Size: {symbol_info.trade_tick_size}")
    print(f"Min Lot: {symbol_info.volume_min}, Lot Step: {symbol_info.volume_step}")
    
    return df, symbol_info

df, symbol_info = download_mt5_data(SYMBOL, TIMEFRAME, NUM_BARS)

# ============================================================
# DOWNLOAD H1 DATA for multi-timeframe features
# ============================================================
def download_h1_data(symbol, num_bars_m15):
    """Download H1 data aligned to M15 timeframe"""
    h1_bars = num_bars_m15 // 4 + 200   # extra bars for EMA warmup
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, h1_bars)
    if rates is None or len(rates) == 0:
        print(f"H1 data download failed: {mt5.last_error()}")
        return None
    
    df_h1 = pd.DataFrame(rates)
    df_h1['time'] = pd.to_datetime(df_h1['time'], unit='s')
    df_h1.set_index('time', inplace=True)
    df_h1.rename(columns={'tick_volume': 'volume'}, inplace=True)
    print(f"H1 data: {len(df_h1)} bars ({df_h1.index[0]} to {df_h1.index[-1]})")
    return df_h1

df_h1 = download_h1_data(SYMBOL, NUM_BARS)

# ============================================================
# AUTO-DETECT PAIR PROPERTIES
# ============================================================
if symbol_info is not None:
    POINT = symbol_info.point           # e.g. 0.01 for gold, 0.00001 for EURUSD
    DIGITS = symbol_info.digits         # e.g. 2 for gold, 5 for EURUSD
    SPREAD_POINTS = symbol_info.spread  # current spread in points
    TICK_VALUE = symbol_info.trade_tick_value   # $ per tick per lot
    TICK_SIZE = symbol_info.trade_tick_size     # price per tick
    CONTRACT_SIZE = symbol_info.trade_contract_size  # contract size
    
    # Fallback: if tick_value=0 (weekend/market closed), compute from contract_size
    if TICK_VALUE == 0 or TICK_VALUE is None:
        TICK_VALUE = TICK_SIZE * CONTRACT_SIZE
        print(f"  WARNING: tick_value=0 (market closed?), using fallback: {TICK_VALUE}")
    
    # Symbol name clean (for file naming)
    SYMBOL_CLEAN = SYMBOL.replace('.', '_').replace(' ', '_')
    
    print(f"\n{'='*50}")
    print(f"Pair Properties (auto-detected):")
    print(f"  POINT       = {POINT}")
    print(f"  DIGITS      = {DIGITS}")
    print(f"  SPREAD      = {SPREAD_POINTS} points ({SPREAD_POINTS * POINT:.5f} price)")
    print(f"  TICK_VALUE  = ${TICK_VALUE} per tick per lot")
    print(f"  TICK_SIZE   = {TICK_SIZE}")
    print(f"  CONTRACT    = {CONTRACT_SIZE}")
    print(f"  File prefix = {SYMBOL_CLEAN}")
    print(f"{'='*50}")

if df is not None:
    print(f"\nSample data:")
    print(df.tail(5))

Symbol XAUUSD.vxc tidak ditemukan!
Available .vxc symbols: []
H1 data download failed: (-1, 'Terminal: Call failed')


## STEP 3: Feature Engineering - Support & Resistance + Trendline Detection
Mendeteksi:
- **Support & Resistance** levels berdasarkan swing highs/lows
- **Trendlines** - garis trend dari swing points (uptrend dari swing lows, downtrend dari swing highs)
- **Jarak price ke S/R dan Trendline** terdekat
- **Retest patterns** (price mendekati S/R level lalu bounce)
- **Trendline break / bounce** detection

- **Market structure** (higher high, lower low, dll)- **Technical indicators** sebagai tambahan

In [4]:
# ============================================================
# SUPPORT & RESISTANCE DETECTION + TRENDLINE FEATURES
# ============================================================

def find_swing_points(df, left=5, right=5):
    """Detect swing highs and swing lows"""
    highs = df['high'].values
    lows = df['low'].values
    n = len(df)
    
    swing_highs = np.full(n, np.nan)
    swing_lows = np.full(n, np.nan)
    
    for i in range(left, n - right):
        # Swing High
        if highs[i] == max(highs[i-left:i+right+1]):
            swing_highs[i] = highs[i]
        # Swing Low
        if lows[i] == min(lows[i-left:i+right+1]):
            swing_lows[i] = lows[i]
    
    return swing_highs, swing_lows


def get_sr_levels(swing_highs, swing_lows, current_idx, max_levels=10):
    """Get nearest S/R levels from swing points"""
    sh = swing_highs[:current_idx]
    sl = swing_lows[:current_idx]
    
    resistance_levels = sh[~np.isnan(sh)][-max_levels:] if np.any(~np.isnan(sh)) else np.array([])
    support_levels = sl[~np.isnan(sl)][-max_levels:] if np.any(~np.isnan(sl)) else np.array([])
    
    return support_levels, resistance_levels


def compute_sr_features(df, swing_highs, swing_lows, max_levels=10):
    """Compute S/R related features for each bar"""
    n = len(df)
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    
    dist_nearest_support = np.full(n, 0.0)
    dist_nearest_resistance = np.full(n, 0.0)
    num_supports_below = np.full(n, 0.0)
    num_resistances_above = np.full(n, 0.0)
    sr_zone_strength = np.full(n, 0.0)
    price_position_in_sr = np.full(n, 0.5)
    retest_support = np.full(n, 0.0)
    retest_resistance = np.full(n, 0.0)
    bounce_from_support = np.full(n, 0.0)
    rejection_from_resistance = np.full(n, 0.0)
    
    for i in range(20, n):
        supports, resistances = get_sr_levels(swing_highs, swing_lows, i, max_levels)
        price = close[i]
        
        if len(supports) == 0 and len(resistances) == 0:
            continue
        
        s_below = supports[supports < price] if len(supports) > 0 else np.array([])
        r_above = resistances[resistances > price] if len(resistances) > 0 else np.array([])
        
        num_supports_below[i] = len(s_below)
        num_resistances_above[i] = len(r_above)
        
        atr = np.mean(high[max(0,i-14):i] - low[max(0,i-14):i]) if i > 14 else 1.0
        if atr == 0: atr = 1.0
        
        if len(s_below) > 0:
            nearest_s = s_below[-1]
            dist_nearest_support[i] = (price - nearest_s) / atr
            if dist_nearest_support[i] < 0.5:
                retest_support[i] = 1.0
                if low[i] <= nearest_s * 1.001 and close[i] > nearest_s:
                    bounce_from_support[i] = 1.0
        
        if len(r_above) > 0:
            nearest_r = r_above[0]
            dist_nearest_resistance[i] = (nearest_r - price) / atr
            if dist_nearest_resistance[i] < 0.5:
                retest_resistance[i] = 1.0
                if high[i] >= nearest_r * 0.999 and close[i] < nearest_r:
                    rejection_from_resistance[i] = 1.0
        
        if len(s_below) > 0 and len(r_above) > 0:
            sr_range = r_above[0] - s_below[-1]
            if sr_range > 0:
                price_position_in_sr[i] = (price - s_below[-1]) / sr_range
        
        zone_threshold = atr * 0.3
        all_sr = np.concatenate([supports, resistances])
        touches = np.sum(np.abs(all_sr - price) < zone_threshold)
        sr_zone_strength[i] = touches
    
    return {
        'dist_nearest_support': dist_nearest_support,
        'dist_nearest_resistance': dist_nearest_resistance,
        'num_supports_below': num_supports_below,
        'num_resistances_above': num_resistances_above,
        'sr_zone_strength': sr_zone_strength,
        'price_position_in_sr': price_position_in_sr,
        'retest_support': retest_support,
        'retest_resistance': retest_resistance,
        'bounce_from_support': bounce_from_support,
        'rejection_from_resistance': rejection_from_resistance,
    }


def compute_trendline_features(df, swing_highs, swing_lows, min_touches=2, lookback_bars=100):
    """Compute trendline features from swing points.
    
    Trendlines:
    - Uptrend line: connecting recent swing lows (ascending)
    - Downtrend line: connecting recent swing highs (descending)
    
    Features:
    - Distance to nearest up/down trendline (normalized by ATR)
    - Trendline slope (normalized)
    - Price above/below trendline
    - Trendline touch/bounce/break signals
    """
    n = len(df)
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    
    # ATR for normalization
    tr = np.maximum(high - low,
                    np.maximum(np.abs(high - np.roll(close, 1)),
                               np.abs(low - np.roll(close, 1))))
    tr[0] = high[0] - low[0]
    atr = pd.Series(tr).rolling(14).mean().values
    
    # Output features
    dist_up_trendline = np.zeros(n)
    dist_down_trendline = np.zeros(n)
    up_trendline_slope = np.zeros(n)
    down_trendline_slope = np.zeros(n)
    price_above_up_tl = np.zeros(n)
    price_below_down_tl = np.zeros(n)
    up_tl_touch = np.zeros(n)
    down_tl_touch = np.zeros(n)
    up_tl_break = np.zeros(n)
    down_tl_break = np.zeros(n)
    tl_squeeze = np.zeros(n)
    
    def fit_trendline(points_x, points_y):
        """Fit a line y = slope*x + intercept using least squares."""
        if len(points_x) < min_touches:
            return None
        x = np.array(points_x, dtype=np.float64)
        y = np.array(points_y, dtype=np.float64)
        n_pts = len(x)
        sum_x = np.sum(x)
        sum_y = np.sum(y)
        sum_xx = np.sum(x * x)
        sum_xy = np.sum(x * y)
        denom = n_pts * sum_xx - sum_x * sum_x
        if abs(denom) < 1e-10:
            return None
        slope = (n_pts * sum_xy - sum_x * sum_y) / denom
        intercept = (sum_y - slope * sum_x) / n_pts
        return slope, intercept
    
    for i in range(lookback_bars, n):
        curr_atr = atr[i]
        if np.isnan(curr_atr) or curr_atr <= 0:
            curr_atr = 1.0
        
        # Collect swing lows in lookback window for uptrend line
        sl_indices = []
        sl_values = []
        for j in range(max(0, i - lookback_bars), i):
            if not np.isnan(swing_lows[j]):
                sl_indices.append(j)
                sl_values.append(swing_lows[j])
        
        # Collect swing highs in lookback window for downtrend line
        sh_indices = []
        sh_values = []
        for j in range(max(0, i - lookback_bars), i):
            if not np.isnan(swing_highs[j]):
                sh_indices.append(j)
                sh_values.append(swing_highs[j])
        
        # --- Uptrend line (from swing lows, positive slope) ---
        if len(sl_indices) >= min_touches:
            use_n = min(5, len(sl_indices))
            sl_x = sl_indices[-use_n:]
            sl_y = sl_values[-use_n:]
            result = fit_trendline(sl_x, sl_y)
            
            if result is not None:
                slope, intercept = result
                tl_value_at_i = slope * i + intercept
                
                if slope > 0:  # valid uptrend (ascending)
                    dist = (close[i] - tl_value_at_i) / curr_atr
                    dist_up_trendline[i] = np.clip(dist, -10, 10)
                    up_trendline_slope[i] = np.clip(slope / curr_atr * 10, -10, 10)
                    price_above_up_tl[i] = 1.0 if close[i] > tl_value_at_i else 0.0
                    
                    if abs(dist) < 0.3:
                        up_tl_touch[i] = 1.0
                    
                    if close[i] < tl_value_at_i and i > 0:
                        prev_tl = slope * (i-1) + intercept
                        if close[i-1] > prev_tl:
                            up_tl_break[i] = 1.0
        
        # --- Downtrend line (from swing highs, negative slope) ---
        if len(sh_indices) >= min_touches:
            use_n = min(5, len(sh_indices))
            sh_x = sh_indices[-use_n:]
            sh_y = sh_values[-use_n:]
            result = fit_trendline(sh_x, sh_y)
            
            if result is not None:
                slope, intercept = result
                tl_value_at_i = slope * i + intercept
                
                if slope < 0:  # valid downtrend (descending)
                    dist = (tl_value_at_i - close[i]) / curr_atr
                    dist_down_trendline[i] = np.clip(dist, -10, 10)
                    down_trendline_slope[i] = np.clip(slope / curr_atr * 10, -10, 10)
                    price_below_down_tl[i] = 1.0 if close[i] < tl_value_at_i else 0.0
                    
                    if abs(dist) < 0.3:
                        down_tl_touch[i] = 1.0
                    
                    if close[i] > tl_value_at_i and i > 0:
                        prev_tl = slope * (i-1) + intercept
                        if close[i-1] < prev_tl:
                            down_tl_break[i] = 1.0
        
        # --- Triangle / Squeeze detection ---
        if dist_up_trendline[i] != 0 and dist_down_trendline[i] != 0:
            total_dist = abs(dist_up_trendline[i]) + abs(dist_down_trendline[i])
            if total_dist < 2.0:
                tl_squeeze[i] = 1.0
    
    features = {
        'dist_up_trendline': dist_up_trendline,
        'dist_down_trendline': dist_down_trendline,
        'up_trendline_slope': up_trendline_slope,
        'down_trendline_slope': down_trendline_slope,
        'price_above_up_tl': price_above_up_tl,
        'price_below_down_tl': price_below_down_tl,
        'up_tl_touch': up_tl_touch,
        'down_tl_touch': down_tl_touch,
        'up_tl_break': up_tl_break,
        'down_tl_break': down_tl_break,
        'tl_squeeze': tl_squeeze,
    }
    
    up_touches = np.sum(up_tl_touch > 0)
    down_touches = np.sum(down_tl_touch > 0)
    up_breaks = np.sum(up_tl_break > 0)
    down_breaks = np.sum(down_tl_break > 0)
    squeezes = np.sum(tl_squeeze > 0)
    print(f"  Trendline features: {len(features)} features")
    print(f"    Up TL touches: {up_touches}, breaks: {up_breaks}")
    print(f"    Down TL touches: {down_touches}, breaks: {down_breaks}")
    print(f"    Squeeze/Triangle bars: {squeezes}")
    
    return features

print("S/R + Trendline detection functions defined ✓")

S/R + Trendline detection functions defined ✓


In [5]:
# ============================================================
# MARKET STRUCTURE & TECHNICAL INDICATORS
# ============================================================

def compute_market_structure(df):
    """Detect market structure: HH, HL, LH, LL"""
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    n = len(df)
    
    # Trend detection via EMA
    ema_fast = pd.Series(close).ewm(span=8).mean().values
    ema_mid = pd.Series(close).ewm(span=21).mean().values
    ema_slow = pd.Series(close).ewm(span=50).mean().values
    
    features = {}
    
    # EMA relationships (normalized)
    features['ema_fast_mid_diff'] = (ema_fast - ema_mid) / ema_mid * 100
    features['ema_mid_slow_diff'] = (ema_mid - ema_slow) / ema_slow * 100
    features['price_above_ema_fast'] = (close > ema_fast).astype(float)
    features['price_above_ema_mid'] = (close > ema_mid).astype(float)
    features['price_above_ema_slow'] = (close > ema_slow).astype(float)
    
    # Higher Highs / Lower Lows detection (rolling window)
    window = 10
    rolling_high = pd.Series(high).rolling(window).max().values
    rolling_low = pd.Series(low).rolling(window).min().values
    
    hh = np.zeros(n)
    ll = np.zeros(n)
    for i in range(window*2, n):
        prev_high = np.max(high[i-window*2:i-window])
        curr_high = np.max(high[i-window:i])
        prev_low = np.min(low[i-window*2:i-window])
        curr_low = np.min(low[i-window:i])
        
        hh[i] = 1.0 if curr_high > prev_high else 0.0
        ll[i] = 1.0 if curr_low < prev_low else 0.0
    
    features['higher_high'] = hh
    features['lower_low'] = ll
    features['trend_score'] = hh - ll  # +1 uptrend, -1 downtrend, 0 ranging
    
    return features


def compute_technical_indicators(df):
    """Compute technical indicators + ADX regime + volume profile"""
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    volume = df['volume'].values.astype(float)
    n = len(df)
    
    features = {}
    
    # ATR (14)
    tr = np.maximum(high - low, 
                    np.maximum(np.abs(high - np.roll(close, 1)),
                              np.abs(low - np.roll(close, 1))))
    tr[0] = high[0] - low[0]
    atr = pd.Series(tr).rolling(14).mean().values
    features['atr_normalized'] = atr / close  # ATR as % of price
    
    # RSI (14)
    delta = np.diff(close, prepend=close[0])
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)
    avg_gain = pd.Series(gain).rolling(14).mean().values
    avg_loss = pd.Series(loss).rolling(14).mean().values
    rs = np.where(avg_loss != 0, avg_gain / avg_loss, 100)
    features['rsi'] = 100 - (100 / (1 + rs))
    
    # Stochastic (14, 3)
    low_14 = pd.Series(low).rolling(14).min().values
    high_14 = pd.Series(high).rolling(14).max().values
    denom = high_14 - low_14
    denom = np.where(denom == 0, 1, denom)
    features['stoch_k'] = ((close - low_14) / denom) * 100
    features['stoch_d'] = pd.Series(features['stoch_k']).rolling(3).mean().values
    
    # MACD
    ema12 = pd.Series(close).ewm(span=12).mean().values
    ema26 = pd.Series(close).ewm(span=26).mean().values
    macd = ema12 - ema26
    signal = pd.Series(macd).ewm(span=9).mean().values
    features['macd_normalized'] = macd / close * 1000
    features['macd_signal_diff'] = (macd - signal) / close * 1000
    features['macd_histogram'] = features['macd_signal_diff']
    
    # Bollinger Bands
    sma20 = pd.Series(close).rolling(20).mean().values
    std20 = pd.Series(close).rolling(20).std().values
    upper_bb = sma20 + 2 * std20
    lower_bb = sma20 - 2 * std20
    bb_width = np.where(sma20 != 0, (upper_bb - lower_bb) / sma20, 0)
    features['bb_position'] = np.where(upper_bb - lower_bb != 0, 
                                        (close - lower_bb) / (upper_bb - lower_bb + 1e-10), 0.5)
    features['bb_width'] = bb_width
    
    # Volume features
    vol_sma = pd.Series(volume).rolling(20).mean().values
    features['volume_ratio'] = np.where(vol_sma != 0, volume / (vol_sma + 1e-10), 1.0)
    
    # Candle patterns
    body = close - df['open'].values
    candle_range = high - low
    features['body_ratio'] = np.where(candle_range != 0, body / (candle_range + 1e-10), 0)
    features['upper_shadow'] = np.where(candle_range != 0, 
        (high - np.maximum(close, df['open'].values)) / (candle_range + 1e-10), 0)
    features['lower_shadow'] = np.where(candle_range != 0,
        (np.minimum(close, df['open'].values) - low) / (candle_range + 1e-10), 0)
    
    # Price momentum (returns)
    features['return_1'] = np.concatenate([[0], np.diff(close) / close[:-1] * 100])
    features['return_3'] = pd.Series(close).pct_change(3).values * 100
    features['return_5'] = pd.Series(close).pct_change(5).values * 100
    features['return_10'] = pd.Series(close).pct_change(10).values * 100
    
    # Volatility
    features['volatility_10'] = pd.Series(features['return_1']).rolling(10).std().values
    features['volatility_20'] = pd.Series(features['return_1']).rolling(20).std().values
    
    # =========================================================
    # ADX - Average Directional Index (Market Regime)
    # ADX > 25 = trending, ADX < 20 = ranging
    # =========================================================
    adx_period = 14
    plus_dm = np.zeros(n)
    minus_dm = np.zeros(n)
    for i in range(1, n):
        up_move = high[i] - high[i-1]
        down_move = low[i-1] - low[i]
        plus_dm[i] = up_move if (up_move > down_move and up_move > 0) else 0
        minus_dm[i] = down_move if (down_move > up_move and down_move > 0) else 0
    
    smoothed_tr = pd.Series(tr).ewm(span=adx_period, adjust=False).mean().values
    smoothed_plus_dm = pd.Series(plus_dm).ewm(span=adx_period, adjust=False).mean().values
    smoothed_minus_dm = pd.Series(minus_dm).ewm(span=adx_period, adjust=False).mean().values
    
    plus_di = np.where(smoothed_tr != 0, 100 * smoothed_plus_dm / smoothed_tr, 0)
    minus_di = np.where(smoothed_tr != 0, 100 * smoothed_minus_dm / smoothed_tr, 0)
    
    dx_denom = plus_di + minus_di
    dx = np.where(dx_denom != 0, 100 * np.abs(plus_di - minus_di) / dx_denom, 0)
    adx = pd.Series(dx).ewm(span=adx_period, adjust=False).mean().values
    
    features['adx'] = adx
    features['plus_di'] = plus_di
    features['minus_di'] = minus_di
    features['di_diff'] = plus_di - minus_di  # positive = bullish trend
    features['is_trending'] = (adx > 25).astype(float)  # 1 if trending
    features['is_ranging'] = (adx < 20).astype(float)   # 1 if ranging
    
    # =========================================================
    # Hurst Exponent (simplified via rescaled range)
    # H > 0.5 = trending, H < 0.5 = mean-reverting, H ≈ 0.5 = random
    # =========================================================
    hurst_window = 50
    hurst = np.full(n, 0.5)
    for i in range(hurst_window, n):
        series = close[i-hurst_window:i]
        mean_val = np.mean(series)
        deviations = series - mean_val
        cumdev = np.cumsum(deviations)
        R = np.max(cumdev) - np.min(cumdev)
        S = np.std(series, ddof=1)
        if S > 0 and R > 0:
            hurst[i] = np.log(R / S) / np.log(hurst_window)
    
    features['hurst_exponent'] = hurst
    features['hurst_trending'] = (hurst > 0.55).astype(float)    # trending regime
    features['hurst_reverting'] = (hurst < 0.45).astype(float)   # mean-reverting regime
    
    # =========================================================
    # Volume Profile / Order Flow Features
    # =========================================================
    # Volume Delta: bullish volume (close > open) - bearish volume (close < open)
    bullish_vol = np.where(close > df['open'].values, volume, 0)
    bearish_vol = np.where(close < df['open'].values, volume, 0)
    vol_delta = bullish_vol - bearish_vol
    
    features['volume_delta'] = vol_delta / (vol_sma + 1e-10)  # normalized
    
    # Cumulative Volume Delta (rolling 10 bars)
    cvd = pd.Series(vol_delta).rolling(10).sum().values
    cvd_norm = cvd / (pd.Series(volume).rolling(10).sum().values + 1e-10)
    features['cvd_10'] = cvd_norm
    
    # Cumulative Volume Delta (rolling 20 bars)
    cvd20 = pd.Series(vol_delta).rolling(20).sum().values
    cvd20_norm = cvd20 / (pd.Series(volume).rolling(20).sum().values + 1e-10)
    features['cvd_20'] = cvd20_norm
    
    # Volume imbalance: ratio of bullish to total volume (rolling 10)
    bull_vol_sum = pd.Series(bullish_vol).rolling(10).sum().values
    total_vol_sum = pd.Series(volume).rolling(10).sum().values
    features['vol_imbalance'] = np.where(total_vol_sum > 0, 
                                          bull_vol_sum / total_vol_sum, 0.5)
    
    # Volume climax: extreme volume with price reversal potential
    vol_z = (volume - vol_sma) / (pd.Series(volume).rolling(20).std().values + 1e-10)
    features['volume_climax'] = (np.abs(vol_z) > 2.0).astype(float)
    
    print(f"  ADX regime: trending={np.mean(adx>25)*100:.0f}%, ranging={np.mean(adx<20)*100:.0f}%")
    print(f"  Hurst avg: {np.nanmean(hurst[hurst_window:]):.3f}")
    
    return features

print("Market structure & indicator functions defined ✓")
print("  + ADX (regime detection), Hurst exponent, Volume profile")

Market structure & indicator functions defined ✓
  + ADX (regime detection), Hurst exponent, Volume profile


## STEP 3b: Smart Money Concepts (SMC) Features
Deteksi SMC structures:
- **Order Blocks (OB)** - Last opposite candle before strong move
- **Fair Value Gaps (FVG)** - Price imbalance/inefficiency
- **Break of Structure (BOS)** - Swing high/low broken
- **Change of Character (CHoCH)** - First trend reversal sign
- **Liquidity Sweeps** - Stop hunts above/below key levels
- **Premium/Discount Zones** - Price relative to swing range

In [6]:
# ============================================================
# SMART MONEY CONCEPTS (SMC) FEATURES
# ============================================================
# Order Blocks, FVG, BOS, CHoCH, Liquidity Sweeps,
# Premium/Discount, Imbalance zones
# ============================================================

def compute_smc_features(df, swing_highs, swing_lows):
    """Compute Smart Money Concepts features"""
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    open_p = df['open'].values
    n = len(df)
    
    # ATR for normalization
    tr = np.maximum(high - low, np.maximum(np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    tr[0] = high[0] - low[0]
    atr = pd.Series(tr).rolling(14).mean().values
    atr = np.where(np.isnan(atr) | (atr < 1e-10), 1.0, atr)
    
    features = {}
    
    # =========================================================
    # 1. ORDER BLOCKS (OB)
    # Last bearish candle before bullish impulse (bullish OB)
    # Last bullish candle before bearish impulse (bearish OB)
    # =========================================================
    bullish_ob_dist = np.zeros(n)        # Distance to nearest bullish OB
    bearish_ob_dist = np.zeros(n)        # Distance to nearest bearish OB
    in_bullish_ob = np.zeros(n)          # Price is inside a bullish OB zone
    in_bearish_ob = np.zeros(n)          # Price is inside a bearish OB zone
    ob_strength = np.zeros(n)            # How strong the impulse was after OB
    
    impulse_threshold = 1.5  # ATR multiplier for "strong" move
    
    # Store OB zones: (ob_high, ob_low, strength, bar_idx)
    bullish_obs = []  # bullish OB = last bearish candle before up move
    bearish_obs = []  # bearish OB = last bullish candle before down move
    
    for i in range(3, n):
        body_i = close[i] - open_p[i]
        body_prev = close[i-1] - open_p[i-1]
        move = (close[i] - close[i-2]) / atr[i]
        
        # Bullish OB: previous candle was bearish, current is strong bullish
        if body_prev < 0 and body_i > 0 and move > impulse_threshold:
            ob_zone = (high[i-1], low[i-1], abs(move), i-1)
            bullish_obs.append(ob_zone)
            # Keep only last 20 OBs
            if len(bullish_obs) > 20:
                bullish_obs.pop(0)
        
        # Bearish OB: previous candle was bullish, current is strong bearish
        if body_prev > 0 and body_i < 0 and move < -impulse_threshold:
            ob_zone = (high[i-1], low[i-1], abs(move), i-1)
            bearish_obs.append(ob_zone)
            if len(bearish_obs) > 20:
                bearish_obs.pop(0)
        
        # Check distance to nearest unmitigated OBs
        for ob_h, ob_l, strength, ob_idx in reversed(bullish_obs):
            dist = (close[i] - (ob_h + ob_l) / 2) / atr[i]
            if abs(dist) < 10:
                bullish_ob_dist[i] = dist
                ob_strength[i] = strength
                if close[i] >= ob_l and close[i] <= ob_h:
                    in_bullish_ob[i] = 1.0
                break
        
        for ob_h, ob_l, strength, ob_idx in reversed(bearish_obs):
            dist = ((ob_h + ob_l) / 2 - close[i]) / atr[i]
            if abs(dist) < 10:
                bearish_ob_dist[i] = dist
                if close[i] >= ob_l and close[i] <= ob_h:
                    in_bearish_ob[i] = 1.0
                break
    
    features['bullish_ob_dist'] = bullish_ob_dist
    features['bearish_ob_dist'] = bearish_ob_dist
    features['in_bullish_ob'] = in_bullish_ob
    features['in_bearish_ob'] = in_bearish_ob
    features['ob_strength'] = ob_strength
    
    # =========================================================
    # 2. FAIR VALUE GAPS (FVG) / Imbalance
    # Bullish FVG: candle[i-2].high < candle[i].low (gap up)
    # Bearish FVG: candle[i-2].low > candle[i].high (gap down)
    # =========================================================
    bullish_fvg_dist = np.zeros(n)
    bearish_fvg_dist = np.zeros(n)
    in_bullish_fvg = np.zeros(n)
    in_bearish_fvg = np.zeros(n)
    fvg_size = np.zeros(n)
    
    bullish_fvgs = []  # (fvg_top, fvg_bottom, size, idx)
    bearish_fvgs = []
    
    for i in range(2, n):
        # Bullish FVG: gap between candle[i-2].high and candle[i].low
        if low[i] > high[i-2]:
            gap = (low[i] - high[i-2]) / atr[i]
            if gap > 0.1:  # minimum gap size
                bullish_fvgs.append((low[i], high[i-2], gap, i))
                if len(bullish_fvgs) > 15:
                    bullish_fvgs.pop(0)
        
        # Bearish FVG: gap between candle[i].high and candle[i-2].low
        if high[i] < low[i-2]:
            gap = (low[i-2] - high[i]) / atr[i]
            if gap > 0.1:
                bearish_fvgs.append((low[i-2], high[i], gap, i))
                if len(bearish_fvgs) > 15:
                    bearish_fvgs.pop(0)
        
        # Check proximity to FVGs
        for fvg_top, fvg_bot, size, fidx in reversed(bullish_fvgs):
            mid = (fvg_top + fvg_bot) / 2
            dist = (close[i] - mid) / atr[i]
            if abs(dist) < 8:
                bullish_fvg_dist[i] = dist
                fvg_size[i] = size
                if close[i] >= fvg_bot and close[i] <= fvg_top:
                    in_bullish_fvg[i] = 1.0
                break
        
        for fvg_top, fvg_bot, size, fidx in reversed(bearish_fvgs):
            mid = (fvg_top + fvg_bot) / 2
            dist = (mid - close[i]) / atr[i]
            if abs(dist) < 8:
                bearish_fvg_dist[i] = dist
                if close[i] >= fvg_bot and close[i] <= fvg_top:
                    in_bearish_fvg[i] = 1.0
                break
    
    features['bullish_fvg_dist'] = bullish_fvg_dist
    features['bearish_fvg_dist'] = bearish_fvg_dist
    features['in_bullish_fvg'] = in_bullish_fvg
    features['in_bearish_fvg'] = in_bearish_fvg
    features['fvg_size'] = fvg_size
    
    # =========================================================
    # 3. BREAK OF STRUCTURE (BOS) & CHANGE OF CHARACTER (CHoCH)
    # BOS: swing high/low broken in direction of trend
    # CHoCH: swing high/low broken AGAINST the trend
    # =========================================================
    bos_bullish = np.zeros(n)    # Recent bullish BOS
    bos_bearish = np.zeros(n)    # Recent bearish BOS
    choch_bullish = np.zeros(n)  # CHoCH to bullish (bear trend broken)
    choch_bearish = np.zeros(n)  # CHoCH to bearish (bull trend broken)
    structure_trend = np.zeros(n)  # +1 bullish, -1 bearish, 0 neutral
    
    # Track recent swing points
    recent_sh = []  # (price, index)
    recent_sl = []
    current_trend = 0  # +1 bull, -1 bear
    
    for i in range(10, n):
        # Collect swing points
        if not np.isnan(swing_highs[i]):
            recent_sh.append((swing_highs[i], i))
            if len(recent_sh) > 10:
                recent_sh.pop(0)
        if not np.isnan(swing_lows[i]):
            recent_sl.append((swing_lows[i], i))
            if len(recent_sl) > 10:
                recent_sl.pop(0)
        
        # Check for BOS/CHoCH
        if len(recent_sh) >= 2 and len(recent_sl) >= 2:
            last_sh = recent_sh[-1][0]
            prev_sh = recent_sh[-2][0] if len(recent_sh) >= 2 else last_sh
            last_sl = recent_sl[-1][0]
            prev_sl = recent_sl[-2][0] if len(recent_sl) >= 2 else last_sl
            
            # Bullish BOS: close breaks above last swing high in uptrend
            if close[i] > last_sh:
                if current_trend >= 0:
                    bos_bullish[i] = 1.0
                else:
                    choch_bullish[i] = 1.0  # Was bearish, now breaking high = CHoCH
                current_trend = 1
            
            # Bearish BOS: close breaks below last swing low in downtrend
            if close[i] < last_sl:
                if current_trend <= 0:
                    bos_bearish[i] = 1.0
                else:
                    choch_bearish[i] = 1.0  # Was bullish, now breaking low = CHoCH
                current_trend = -1
        
        structure_trend[i] = current_trend
    
    features['bos_bullish'] = bos_bullish
    features['bos_bearish'] = bos_bearish
    features['choch_bullish'] = choch_bullish
    features['choch_bearish'] = choch_bearish
    features['structure_trend'] = structure_trend
    
    # =========================================================
    # 4. LIQUIDITY SWEEPS
    # Price takes out previous high/low then reverses
    # =========================================================
    liq_sweep_high = np.zeros(n)   # Swept above previous highs
    liq_sweep_low = np.zeros(n)    # Swept below previous lows
    liq_grab_bullish = np.zeros(n) # Swept low then closed above (bullish)
    liq_grab_bearish = np.zeros(n) # Swept high then closed below (bearish)
    
    lookback_liq = 20
    
    for i in range(lookback_liq + 1, n):
        # Previous highs and lows in lookback window
        prev_high = np.max(high[i-lookback_liq:i])
        prev_low = np.min(low[i-lookback_liq:i])
        
        # Sweep above: wick went above prev high but closed below
        if high[i] > prev_high and close[i] < prev_high:
            liq_sweep_high[i] = 1.0
            if close[i] < open_p[i]:  # Bearish close after sweep = bearish grab
                liq_grab_bearish[i] = 1.0
        
        # Sweep below: wick went below prev low but closed above
        if low[i] < prev_low and close[i] > prev_low:
            liq_sweep_low[i] = 1.0
            if close[i] > open_p[i]:  # Bullish close after sweep = bullish grab
                liq_grab_bullish[i] = 1.0
    
    features['liq_sweep_high'] = liq_sweep_high
    features['liq_sweep_low'] = liq_sweep_low
    features['liq_grab_bullish'] = liq_grab_bullish
    features['liq_grab_bearish'] = liq_grab_bearish
    
    # =========================================================
    # 5. PREMIUM / DISCOUNT ZONES
    # Premium = above 50% of recent range (sell zone)
    # Discount = below 50% of recent range (buy zone)
    # =========================================================
    premium_discount = np.zeros(n)  # -1 to +1 (-1=deep discount, +1=deep premium)
    in_premium = np.zeros(n)
    in_discount = np.zeros(n)
    
    range_window = 50
    for i in range(range_window, n):
        range_high = np.max(high[i-range_window:i])
        range_low = np.min(low[i-range_window:i])
        range_size = range_high - range_low
        
        if range_size > 0:
            position = (close[i] - range_low) / range_size  # 0 to 1
            premium_discount[i] = position * 2 - 1  # -1 to +1
            
            # Fibonacci zones
            # Premium zone: above 0.5 (preferably above 0.618)
            # Discount zone: below 0.5 (preferably below 0.382)
            if position > 0.618:
                in_premium[i] = 1.0
            elif position < 0.382:
                in_discount[i] = 1.0
    
    features['premium_discount'] = premium_discount
    features['in_premium'] = in_premium
    features['in_discount'] = in_discount
    
    # =========================================================
    # 6. DISPLACEMENT / IMPULSE CANDLES
    # Large body candles that show institutional activity
    # =========================================================
    displacement_up = np.zeros(n)
    displacement_down = np.zeros(n)
    displacement_strength = np.zeros(n)
    
    for i in range(1, n):
        body = close[i] - open_p[i]
        body_atr = abs(body) / atr[i]
        
        # Displacement = body > 1.5 ATR
        if body_atr > 1.5:
            displacement_strength[i] = body_atr
            if body > 0:
                displacement_up[i] = 1.0
            else:
                displacement_down[i] = 1.0
    
    features['displacement_up'] = displacement_up
    features['displacement_down'] = displacement_down
    features['displacement_strength'] = displacement_strength
    
    # =========================================================
    # 7. SMC CONFLUENCE SCORE
    # Combine multiple SMC signals into a single score
    # =========================================================
    smc_bullish_score = np.zeros(n)
    smc_bearish_score = np.zeros(n)
    
    for i in range(1, n):
        bull_score = 0
        bear_score = 0
        
        # Order Block confluence
        if in_bullish_ob[i]: bull_score += 2
        if in_bearish_ob[i]: bear_score += 2
        
        # FVG confluence
        if in_bullish_fvg[i]: bull_score += 1.5
        if in_bearish_fvg[i]: bear_score += 1.5
        
        # Structure confluence
        if bos_bullish[i]: bull_score += 1
        if bos_bearish[i]: bear_score += 1
        if choch_bullish[i]: bull_score += 2
        if choch_bearish[i]: bear_score += 2
        
        # Liquidity
        if liq_grab_bullish[i]: bull_score += 2
        if liq_grab_bearish[i]: bear_score += 2
        
        # Premium/Discount
        if in_discount[i]: bull_score += 1
        if in_premium[i]: bear_score += 1
        
        # Displacement
        if displacement_up[i]: bull_score += 1
        if displacement_down[i]: bear_score += 1
        
        smc_bullish_score[i] = bull_score
        smc_bearish_score[i] = bear_score
    
    features['smc_bullish_score'] = smc_bullish_score
    features['smc_bearish_score'] = smc_bearish_score
    features['smc_net_score'] = smc_bullish_score - smc_bearish_score
    
    print(f"SMC Features computed: {len(features)} features")
    print(f"  Order Blocks found: {int(np.sum(in_bullish_ob > 0))} bullish, {int(np.sum(in_bearish_ob > 0))} bearish")
    print(f"  FVGs found: {int(np.sum(in_bullish_fvg > 0))} bullish, {int(np.sum(in_bearish_fvg > 0))} bearish")
    print(f"  BOS events: {int(np.sum(bos_bullish))} bullish, {int(np.sum(bos_bearish))} bearish")
    print(f"  CHoCH events: {int(np.sum(choch_bullish))} bullish, {int(np.sum(choch_bearish))} bearish")
    print(f"  Liquidity grabs: {int(np.sum(liq_grab_bullish))} bullish, {int(np.sum(liq_grab_bearish))} bearish")
    
    return features

print("SMC feature functions defined ✓")

SMC feature functions defined ✓


In [10]:
# ============================================================
# DATASET V6: S/R + SMC + Technical + ADX/Hurst + Volume + 
#             Multi-TF(H1) + Time + 3-class (SELL/BUY/HOLD)
# ============================================================

def compute_h1_context_features(df_m15, df_h1_raw):
    """Compute H1 timeframe context features aligned to M15 bars.
    For each M15 bar, compute H1 trend/RSI/ATR from the enclosing H1 bar."""
    n = len(df_m15)
    features = {}
    
    if df_h1_raw is None or len(df_h1_raw) == 0:
        # Fallback: all zeros if H1 data not available
        for name in ['h1_ema_trend', 'h1_rsi', 'h1_atr_ratio', 'h1_body_ratio',
                      'h1_trend_score', 'h1_bb_position', 'h1_volume_ratio']:
            features[name] = np.zeros(n)
        print("  H1 context: SKIPPED (no data)")
        return features
    
    h1_close = df_h1_raw['close'].values
    h1_high = df_h1_raw['high'].values
    h1_low = df_h1_raw['low'].values
    h1_open = df_h1_raw['open'].values
    h1_vol = df_h1_raw['volume'].values.astype(float)
    nh1 = len(df_h1_raw)
    
    # H1 indicators
    h1_ema8 = pd.Series(h1_close).ewm(span=8).mean().values
    h1_ema21 = pd.Series(h1_close).ewm(span=21).mean().values
    h1_ema50 = pd.Series(h1_close).ewm(span=50).mean().values
    
    # H1 RSI
    h1_delta = np.diff(h1_close, prepend=h1_close[0])
    h1_gain = np.where(h1_delta > 0, h1_delta, 0)
    h1_loss = np.where(h1_delta < 0, -h1_delta, 0)
    h1_avg_gain = pd.Series(h1_gain).rolling(14).mean().values
    h1_avg_loss = pd.Series(h1_loss).rolling(14).mean().values
    h1_rs = np.where(h1_avg_loss != 0, h1_avg_gain / h1_avg_loss, 100)
    h1_rsi = 100 - (100 / (1 + h1_rs))
    
    # H1 ATR
    h1_tr = np.maximum(h1_high - h1_low,
                        np.maximum(np.abs(h1_high - np.roll(h1_close, 1)),
                                   np.abs(h1_low - np.roll(h1_close, 1))))
    h1_tr[0] = h1_high[0] - h1_low[0]
    h1_atr = pd.Series(h1_tr).rolling(14).mean().values
    
    # H1 BB position
    h1_sma20 = pd.Series(h1_close).rolling(20).mean().values
    h1_std20 = pd.Series(h1_close).rolling(20).std().values
    h1_upper_bb = h1_sma20 + 2 * h1_std20
    h1_lower_bb = h1_sma20 - 2 * h1_std20
    h1_bb_pos = np.where(h1_upper_bb - h1_lower_bb != 0,
                          (h1_close - h1_lower_bb) / (h1_upper_bb - h1_lower_bb + 1e-10), 0.5)
    
    # H1 volume SMA
    h1_vol_sma = pd.Series(h1_vol).rolling(20).mean().values
    
    # Map H1 bars to M15 index via searchsorted
    h1_times = df_h1_raw.index.values
    m15_times = df_m15.index.values
    h1_idx_map = np.searchsorted(h1_times, m15_times, side='right') - 1
    h1_idx_map = np.clip(h1_idx_map, 0, nh1 - 1)
    
    # Build features
    h1_ema_trend = np.zeros(n)
    h1_rsi_feat = np.zeros(n)
    h1_atr_ratio_feat = np.zeros(n)
    h1_body_ratio_feat = np.zeros(n)
    h1_trend_score_feat = np.zeros(n)
    h1_bb_pos_feat = np.zeros(n)
    h1_vol_ratio_feat = np.zeros(n)
    
    for i in range(n):
        hi = h1_idx_map[i]
        if hi < 50:  # need warmup
            continue
        
        # H1 EMA trend: (ema8 - ema21) / ema21 * 100
        h1_ema_trend[i] = (h1_ema8[hi] - h1_ema21[hi]) / (h1_ema21[hi] + 1e-10) * 100
        h1_rsi_feat[i] = h1_rsi[hi]
        h1_atr_ratio_feat[i] = h1_atr[hi] / (h1_close[hi] + 1e-10)
        
        h1_range = h1_high[hi] - h1_low[hi]
        h1_body = h1_close[hi] - h1_open[hi]
        h1_body_ratio_feat[i] = h1_body / (h1_range + 1e-10) if h1_range > 0 else 0
        
        # H1 HH/LL
        if hi >= 20:
            ph = np.max(h1_high[hi-20:hi-10])
            ch = np.max(h1_high[hi-10:hi])
            pl = np.min(h1_low[hi-20:hi-10])
            cl = np.min(h1_low[hi-10:hi])
            h1_hh = 1.0 if ch > ph else 0.0
            h1_ll = 1.0 if cl < pl else 0.0
            h1_trend_score_feat[i] = h1_hh - h1_ll
        
        h1_bb_pos_feat[i] = h1_bb_pos[hi]
        h1_vol_ratio_feat[i] = h1_vol[hi] / (h1_vol_sma[hi] + 1e-10) if h1_vol_sma[hi] > 0 else 1.0
    
    features['h1_ema_trend'] = h1_ema_trend
    features['h1_rsi'] = h1_rsi_feat
    features['h1_atr_ratio'] = h1_atr_ratio_feat
    features['h1_body_ratio'] = h1_body_ratio_feat
    features['h1_trend_score'] = h1_trend_score_feat
    features['h1_bb_position'] = h1_bb_pos_feat
    features['h1_volume_ratio'] = h1_vol_ratio_feat
    
    print(f"  H1 context features: {len(features)} features mapped to M15")
    return features


def create_dataset_v5(df, future_bars=8, df_h1_data=None):
    """Create dataset with 3-class labels (SELL/BUY/HOLD) + all features"""
    close = df['close'].values
    high = df['high'].values
    low = df['low'].values
    open_p = df['open'].values
    n = len(df)
    
    # Spread dari data MT5 (kolom 'spread' = spread in points per bar)
    if 'spread' in df.columns:
        spread_price = df['spread'].values * POINT
        avg_spread_pts = df['spread'].values[df['spread'].values > 0].mean()
        avg_spread_price = avg_spread_pts * POINT
        print(f"Spread from MT5 data: avg={avg_spread_pts:.0f} points = {avg_spread_price:.5f} price")
    else:
        spread_price = np.full(n, SPREAD_POINTS * POINT)
        avg_spread_price = SPREAD_POINTS * POINT
        print(f"Spread from symbol_info: {SPREAD_POINTS} points = {avg_spread_price:.5f} price")
    
    # ATR
    tr = np.maximum(high - low, np.maximum(np.abs(high - np.roll(close, 1)), np.abs(low - np.roll(close, 1))))
    tr[0] = high[0] - low[0]
    atr = pd.Series(tr).rolling(14).mean().values
    df['atr'] = atr
    
    print("Computing S/R features...")
    swing_highs, swing_lows = find_swing_points(df, left=5, right=5)
    sr_features = compute_sr_features(df, swing_highs, swing_lows)
    
    print("Computing Trendline features...")
    tl_features = compute_trendline_features(df, swing_highs, swing_lows)
    
    print("Computing Market Structure features...")
    ms_features = compute_market_structure(df)
    
    print("Computing Technical + ADX + Volume features...")
    ti_features = compute_technical_indicators(df)
    
    print("Computing SMC features...")
    smc_features = compute_smc_features(df, swing_highs, swing_lows)
    
    # Combine all features (ORDER MATTERS - must match MQL5 EA)
    all_features = {}
    all_features.update(sr_features)       # 10 features
    all_features.update(tl_features)       # 11 trendline features
    all_features.update(ms_features)       # 8 features
    all_features.update(ti_features)       # 19 + 12 new = 31 features (ADX/Hurst/VolProfile)
    
    # Candle features
    all_features['candle_body_pct'] = (close - open_p) / (atr + 1e-10)
    all_features['candle_range_pct'] = (high - low) / (atr + 1e-10)
    
    all_features.update(smc_features)      # 28 features
    
    # ---- TIME-BASED FEATURES ----
    print("Computing Time-Based features...")
    hours = df.index.hour.values
    day_of_week = df.index.dayofweek.values
    
    all_features['hour_sin'] = np.sin(2 * np.pi * hours / 24.0)
    all_features['hour_cos'] = np.cos(2 * np.pi * hours / 24.0)
    all_features['day_sin'] = np.sin(2 * np.pi * day_of_week / 5.0)
    all_features['day_cos'] = np.cos(2 * np.pi * day_of_week / 5.0)
    all_features['session_asian'] = ((hours >= 0) & (hours < 8)).astype(float)
    all_features['session_london'] = ((hours >= 7) & (hours < 16)).astype(float)
    all_features['session_newyork'] = ((hours >= 13) & (hours < 22)).astype(float)
    all_features['session_overlap'] = ((hours >= 13) & (hours < 16)).astype(float)
    all_features['is_high_activity'] = (
        ((hours >= 8) & (hours < 11)) | ((hours >= 13) & (hours < 16))
    ).astype(float)
    
    # ---- MULTI-TIMEFRAME H1 CONTEXT ----
    print("Computing H1 multi-timeframe context...")
    h1_features = compute_h1_context_features(df, df_h1_data)
    all_features.update(h1_features)       # 7 features
    
    feature_df = pd.DataFrame(all_features, index=df.index)
    
    # ---- 3-CLASS LABELING: SELL(0) / BUY(1) / HOLD(2) ----
    # HOLD = situations where neither BUY nor SELL has clear edge
    print("Creating 3-class labels (SELL/BUY/HOLD)...")
    labels = np.full(n, -1, dtype=np.int64)
    sl_mult = 2.5
    rr_ratio = 2.0
    
    buy_wins = 0
    sell_wins = 0
    hold_count = 0
    
    for i in range(n - future_bars * 4):
        current_atr = atr[i]
        if np.isnan(current_atr) or current_atr <= 0:
            continue
        
        bar_spread = spread_price[i] if i < len(spread_price) else avg_spread_price
        half_spread = bar_spread / 2.0
        
        sl_dist = current_atr * sl_mult
        tp_dist = sl_dist * rr_ratio
        
        # --- Simulate BUY ---
        buy_entry = close[i] + half_spread
        buy_sl = buy_entry - sl_dist
        buy_tp = buy_entry + tp_dist
        
        buy_result = 0
        for j in range(1, future_bars * 4):
            if i + j >= n:
                break
            if low[i+j] <= buy_sl:
                buy_result = -1
                break
            if high[i+j] >= buy_tp:
                buy_result = 1
                break
        
        # --- Simulate SELL ---
        sell_entry = close[i] - half_spread
        sell_sl = sell_entry + sl_dist
        sell_tp = sell_entry - tp_dist
        
        sell_result = 0
        for j in range(1, future_bars * 4):
            if i + j >= n:
                break
            if high[i+j] >= sell_sl:
                sell_result = -1
                break
            if low[i+j] <= sell_tp:
                sell_result = 1
                break
        
        # 3-class labeling logic
        if buy_result == 1 and sell_result != 1:
            labels[i] = 1  # BUY
            buy_wins += 1
        elif sell_result == 1 and buy_result != 1:
            labels[i] = 0  # SELL
            sell_wins += 1
        elif buy_result == -1 and sell_result == -1:
            # Both directions hit SL → HOLD (unclear market)
            labels[i] = 2  # HOLD
            hold_count += 1
        elif buy_result == 0 and sell_result == 0:
            # Neither direction resolved → HOLD (low volatility / choppy)
            labels[i] = 2  # HOLD
            hold_count += 1
        elif buy_result == 1 and sell_result == 1:
            # Both won → HOLD (too volatile, unreliable)
            labels[i] = 2
            hold_count += 1
    
    feature_df['label'] = labels
    feature_df = feature_df.iloc[60:-future_bars*4].copy()
    feature_df_full = feature_df.copy()
    
    # Only labeled samples (now includes label=2 for HOLD)
    labeled = feature_df[feature_df['label'] >= 0].copy().dropna()
    for col in labeled.columns:
        if col != 'label':
            labeled[col] = labeled[col].clip(-10, 10)
    for col in feature_df_full.columns:
        if col != 'label':
            feature_df_full[col] = feature_df_full[col].clip(-10, 10)
    
    avg_atr = np.nanmean(atr[60:])
    print(f"\nTotal bars: {len(feature_df_full)}")
    print(f"Labeled: {len(labeled)} ({len(labeled)/len(feature_df_full)*100:.1f}%)")
    print(f"SELL: {(labeled['label']==0).sum()} ({(labeled['label']==0).mean()*100:.1f}%)")
    print(f"BUY:  {(labeled['label']==1).sum()} ({(labeled['label']==1).mean()*100:.1f}%)")
    print(f"HOLD: {(labeled['label']==2).sum()} ({(labeled['label']==2).mean()*100:.1f}%)")
    print(f"Avg spread: {avg_spread_price:.5f} ({avg_spread_price/avg_atr*100:.1f}% of ATR)")
    print(f"\nBase features: {len([c for c in labeled.columns if c != 'label'])}")

    return labeled, feature_df_full

feature_df, feature_df_full = create_dataset_v5(df, future_bars=8, df_h1_data=df_h1)

TypeError: 'NoneType' object is not subscriptable

## STEP 4: Prepare Data for XGBoost
Scaling features dan menyiapkan numpy arrays untuk XGBoost training.

In [ ]:
# ============================================================
# PREPARE DATA FOR XGBOOST - With lag features + 3-class support
# ============================================================

def prepare_data_xgb(feature_df, test_ratio=0.15, val_ratio=0.15):
    """Prepare data with lag features for XGBoost (3-class: SELL/BUY/HOLD)"""
    
    feature_cols = [c for c in feature_df.columns if c != 'label']
    
    df_with_lags = feature_df.copy()
    
    # Core lag features (S/R + Technical + SMC)
    lag_cols = [
        'dist_nearest_support', 'dist_nearest_resistance', 'rsi',
        'macd_normalized', 'return_1', 'trend_score', 'bb_position',
        'price_position_in_sr', 'candle_body_pct',
        # SMC features for lags
        'smc_net_score', 'structure_trend', 'premium_discount',
        'bullish_ob_dist', 'bearish_ob_dist',
    ]
    
    for lag in [1, 2, 3, 5]:
        for col in lag_cols:
            if col in df_with_lags.columns:
                df_with_lags[f'{col}_lag{lag}'] = df_with_lags[col].shift(lag)
    
    # Change features (momentum)
    change_cols = [
        'dist_nearest_support', 'rsi', 'macd_normalized',
        'price_position_in_sr', 'smc_net_score', 'premium_discount',
    ]
    for col in change_cols:
        if col in df_with_lags.columns:
            df_with_lags[f'{col}_change'] = df_with_lags[col].diff()
    
    df_with_lags = df_with_lags.dropna()
    
    feature_cols_all = [c for c in df_with_lags.columns if c != 'label']
    X = df_with_lags[feature_cols_all].values.astype(np.float32)
    y = df_with_lags['label'].values.astype(np.int64)
    
    X = np.clip(X, -10, 10)
    
    # Chronological split
    n = len(X)
    test_start = int(n * (1 - test_ratio))
    val_start = int(n * (1 - test_ratio - val_ratio))
    
    X_train, y_train = X[:val_start], y[:val_start]
    X_val, y_val = X[val_start:test_start], y[val_start:test_start]
    X_test, y_test = X[test_start:], y[test_start:]
    
    # RobustScaler
    scaler = RobustScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    
    # Class distribution (3-class)
    num_classes = len(np.unique(y_train))
    print(f"Features (with lags): {len(feature_cols_all)}")
    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    print(f"Classes: {num_classes}")
    for c in range(num_classes):
        label_name = {0: 'SELL', 1: 'BUY', 2: 'HOLD'}.get(c, str(c))
        print(f"  {label_name}: Train={np.mean(y_train==c)*100:.1f}% Val={np.mean(y_val==c)*100:.1f}% Test={np.mean(y_test==c)*100:.1f}%")
    
    return (X_train_s, y_train, X_val_s, y_val, X_test_s, y_test,
            scaler, feature_cols_all)

(X_train, y_train, X_val, y_val, X_test, y_test,
 scaler, feature_cols) = prepare_data_xgb(feature_df)
NUM_FEATURES = len(feature_cols)
print(f"\nTotal features: {NUM_FEATURES}")

## STEP 5: Build & Train XGBoost Model
Model: **XGBoost Gradient Boosting**
- Tree-based ensemble untuk menangkap pola non-linear
- Built-in feature importance
- Faster training, optional GPU acceleration
- Multi-class classification: SELL/BUY/HOLD

In [ ]:
# ============================================================
# XGBOOST MODEL - 3-class Classification (SELL/BUY/HOLD)
# Anti-Overfitting Version
# ============================================================

def train_xgboost_model(X_train, y_train, X_val, y_val,
                         n_estimators=1500, learning_rate=0.03,
                         max_depth=4, early_stopping_rounds=50):
    """Train XGBoost model with early stopping and class weighting"""
    
    start = time.time()
    
    # Compute sample weights for class imbalance
    num_classes = len(np.unique(y_train))
    class_counts = np.bincount(y_train, minlength=num_classes).astype(float)
    class_weights = len(y_train) / (num_classes * class_counts + 1e-6)
    sample_weights = np.array([class_weights[label] for label in y_train])
    
    print(f"Class counts: {class_counts.astype(int)}")
    print(f"Class weights: {class_weights.round(3)}")
    
    # XGBoost parameters - ANTI-OVERFITTING tuning
    params = {
        'objective': 'multi:softprob',
        'num_class': num_classes,
        'eval_metric': 'mlogloss',
        'max_depth': max_depth,              # 4 instead of 8 (much less memorization)
        'learning_rate': learning_rate,       # 0.03 instead of 0.05 (slower learning)
        'n_estimators': n_estimators,
        'subsample': 0.6,                    # 0.6 instead of 0.8 (more row dropout)
        'colsample_bytree': 0.6,             # 0.6 instead of 0.8 (more feature dropout)
        'colsample_bylevel': 0.5,            # 0.5 instead of 0.7
        'colsample_bynode': 0.8,             # NEW: additional column sampling per split
        'min_child_weight': 20,              # 20 instead of 5 (requires more samples per leaf)
        'gamma': 1.0,                        # 1.0 instead of 0.1 (stronger pruning)
        'reg_alpha': 2.0,                    # 2.0 instead of 0.5 (more L1 regularization)
        'reg_lambda': 5.0,                   # 5.0 instead of 2.0 (more L2 regularization)
        'max_delta_step': 1,                 # Helps with imbalanced classes
        'scale_pos_weight': 1,
        'tree_method': 'hist',               # Fast histogram-based method
        'max_leaves': 31,                    # NEW: limit leaf nodes (default is 0=unlimited)
        'early_stopping_rounds': early_stopping_rounds,  # Stop when val loss stops improving
        'random_state': 42,
        'verbosity': 0,
    }
    
    # Try GPU if available
    try:
        test_model = xgb.XGBClassifier(tree_method='gpu_hist', n_estimators=1, verbosity=0)
        test_model.fit(X_train[:100], y_train[:100])
        params['tree_method'] = 'gpu_hist'
        params['gpu_id'] = 0
        print("Using GPU acceleration for XGBoost!")
    except Exception:
        print("Using CPU for XGBoost (GPU not available)")
    
    model = xgb.XGBClassifier(**params)
    
    print(f"\nTraining XGBoost with {n_estimators} max trees...")
    print(f"  max_depth={max_depth}, lr={learning_rate}, early_stop={early_stopping_rounds}")
    model.fit(
        X_train, y_train,
        sample_weight=sample_weights,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=100
    )
    
    elapsed = time.time() - start
    
    # Results - use best_iteration from early stopping
    best_iteration = model.best_iteration if hasattr(model, 'best_iteration') and model.best_iteration is not None else model.n_estimators
    
    train_probs = model.predict_proba(X_train)
    val_probs = model.predict_proba(X_val)
    train_preds = np.argmax(train_probs, axis=1)
    val_preds = np.argmax(val_probs, axis=1)
    
    train_acc = np.mean(train_preds == y_train)
    val_acc = np.mean(val_preds == y_val)
    
    # High confidence actionable accuracy
    val_max_conf = np.max(val_probs, axis=1)
    actionable_mask = val_preds != 2
    hc_mask = (val_max_conf >= 0.55) & actionable_mask
    hc_acc = np.mean(val_preds[hc_mask] == y_val[hc_mask]) if hc_mask.sum() > 20 else 0
    
    print(f"\n{'='*60}")
    print(f"Training done in {elapsed:.1f}s")
    print(f"Best iteration: {best_iteration} / {n_estimators}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Gap: {train_acc-val_acc:.4f}")
    print(f"HC55 Acc: {hc_acc:.4f} ({hc_mask.sum()} signals)")
    print(f"{'='*60}")
    
    return model

model = train_xgboost_model(
    X_train, y_train, X_val, y_val,
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=4,
    early_stopping_rounds=50
)

# Save model
model.save_model('best_xgb_model_v5.json')
print("Model saved: best_xgb_model_v5.json")
# Save feature order (WAJIB untuk inferensi engine)
import json
with open("feature_cols_v5.json", "w") as f:
    json.dump(list(feature_cols), f, indent=2)
print("Feature cols saved: feature_cols_v5.json", len(feature_cols))

## STEP 5b: Feature Importance Analysis
Analisis fitur mana yang paling berpengaruh terhadap prediksi model XGBoost.

In [ ]:
# ============================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================

importance = model.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
}).sort_values('importance', ascending=False)

print("Top 30 Most Important Features:")
print("=" * 55)
for idx, row in importance_df.head(30).iterrows():
    bar = '\u2588' * int(row['importance'] * 500)
    print(f"  {row['feature']:<35} {row['importance']:.4f} {bar}")

# Plot top 30
fig, ax = plt.subplots(figsize=(12, 8))
top30 = importance_df.head(30)
ax.barh(range(len(top30)), top30['importance'].values, color='steelblue')
ax.set_yticks(range(len(top30)))
ax.set_yticklabels(top30['feature'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('XGBoost Feature Importance (Top 30)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT TRAINING HISTORY
# ============================================================

results_log = model.evals_result()
epochs_range = range(len(results_log['validation_0']['mlogloss']))

fig, ax = plt.subplots(1, 1, figsize=(12, 5))

ax.plot(list(epochs_range), results_log['validation_0']['mlogloss'], label='Train Loss', alpha=0.8)
ax.plot(list(epochs_range), results_log['validation_1']['mlogloss'], label='Val Loss', alpha=0.8)
ax.set_title('XGBoost Training Loss (mlogloss)')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('Log Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TEST SET EVALUATION (3-class: SELL/BUY/HOLD)
# ============================================================

all_probs = model.predict_proba(X_test)
all_preds = np.argmax(all_probs, axis=1)
all_labels = y_test

print("=" * 60)
print("TEST SET EVALUATION (3-Class: SELL=0, BUY=1, HOLD=2)")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=['SELL', 'BUY', 'HOLD']))
print("Confusion Matrix (rows=actual, cols=predicted):")
print("          SELL  BUY  HOLD")
cm = confusion_matrix(all_labels, all_preds)
for i, name in enumerate(['SELL', 'BUY ', 'HOLD']):
    print(f"  {name}: {cm[i]}")

# Actionable accuracy (only BUY/SELL, ignore HOLD predictions)
actionable_mask = all_preds != 2
if actionable_mask.sum() > 0:
    actionable_acc = np.mean(all_preds[actionable_mask] == all_labels[actionable_mask])
    print(f"\nActionable predictions (BUY/SELL only): {actionable_mask.sum()} ({actionable_mask.mean()*100:.1f}%)")
    print(f"Actionable accuracy: {actionable_acc*100:.1f}%")

# Confidence analysis for actionable signals
print("\nConfidence analysis (BUY/SELL only, ignoring HOLD):")
for threshold in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75]:
    conf = np.max(all_probs, axis=1)
    mask = (conf >= threshold) & (all_preds != 2)  # high conf + actionable
    if mask.sum() > 0:
        acc = np.mean(all_preds[mask] == all_labels[mask])
        pct_buy = np.mean(all_preds[mask] == 1) * 100
        pct_sell = np.mean(all_preds[mask] == 0) * 100
        print(f"  Conf >= {threshold}: {mask.sum()} signals ({mask.mean()*100:.1f}%), "
              f"Acc: {acc*100:.1f}%, BUY: {pct_buy:.0f}% SELL: {pct_sell:.0f}%")
    else:
        print(f"  Conf >= {threshold}: 0 signals")

# HOLD rate
hold_rate = np.mean(all_preds == 2) * 100
print(f"\nHOLD rate: {hold_rate:.1f}% (model avoids {hold_rate:.0f}% of bars)")

## STEP 7: Backtest Strategy
Simulasi trading menggunakan prediksi model pada test data.
- Entry berdasarkan sinyal model (confidence > threshold)
- SL/TP dinamis dari model prediction
- Menghitung: Win Rate, Profit Factor, Max Drawdown, Sharpe Ratio

In [ ]:
# ============================================================
# BACKTEST V5 - Universal + Slippage + 3-class (skip HOLD)
# ============================================================

def backtest_xgb(model, feature_df_full, df_full, scaler, feature_cols,
                rr_ratio=2.0, sl_atr_mult=1.5, confidence_threshold=0.52,
                lot_size=0.01, max_bars=48, slippage_points=2):
    """Backtest universal - spread + slippage from data, PnL from tick_value.
    3-class model: only trades BUY(1) or SELL(0), skips HOLD(2)."""
    
    # Build features with lags (same as prepare_data_v5)
    df_bt = feature_df_full.copy()
    
    lag_cols = [
        'dist_nearest_support', 'dist_nearest_resistance', 'rsi',
        'macd_normalized', 'return_1', 'trend_score', 'bb_position',
        'price_position_in_sr', 'candle_body_pct',
        'smc_net_score', 'structure_trend', 'premium_discount',
        'bullish_ob_dist', 'bearish_ob_dist',
    ]
    for lag in [1, 2, 3, 5]:
        for col in lag_cols:
            if col in df_bt.columns:
                df_bt[f'{col}_lag{lag}'] = df_bt[col].shift(lag)
    
    change_cols = [
        'dist_nearest_support', 'rsi', 'macd_normalized',
        'price_position_in_sr', 'smc_net_score', 'premium_discount',
    ]
    for col in change_cols:
        if col in df_bt.columns:
            df_bt[f'{col}_change'] = df_bt[col].diff()
    
    df_bt = df_bt.dropna()
    valid_idx = df_bt.index
    
    X_all = df_bt[feature_cols].values.astype(np.float32)
    X_all = np.clip(X_all, -10, 10)
    X_scaled = scaler.transform(X_all)
    
    # Get predictions using XGBoost
    all_probs = model.predict_proba(X_scaled)
    
    # Backtest on TEST portion (last 15%)
    n_total = len(all_probs)
    test_start = int(n_total * 0.7)
    
    ts_to_pos = {ts: pos for pos, ts in enumerate(df_full.index)}
    pnl_per_point = TICK_VALUE / TICK_SIZE
    spread_col = df_full['spread'].values * POINT if 'spread' in df_full.columns else np.full(len(df_full), SPREAD_POINTS * POINT)
    slippage_price = slippage_points * POINT
    
    trades = []
    equity = 10000.0
    peak_equity = equity
    max_dd = 0
    skipped_hold = 0
    
    for i in range(test_start, n_total - 1):
        # 3-class: probs = [prob_sell, prob_buy, prob_hold]
        prob_sell = all_probs[i][0]
        prob_buy = all_probs[i][1]
        prob_hold = all_probs[i][2] if all_probs.shape[1] > 2 else 0
        
        # Skip if HOLD is the highest class
        pred_class = np.argmax(all_probs[i])
        if pred_class == 2:
            skipped_hold += 1
            continue
        
        max_prob = max(prob_sell, prob_buy)
        if max_prob < confidence_threshold:
            continue
        
        signal = 1 if prob_buy > prob_sell else 0
        
        bar_ts = valid_idx[i]
        bar_pos = ts_to_pos.get(bar_ts, None)
        if bar_pos is None or bar_pos >= len(df_full) - 1:
            continue
        
        entry_price = df_full['close'].iloc[bar_pos]
        atr = df_full['atr'].iloc[bar_pos] if 'atr' in df_full.columns else None
        if atr is None or atr < POINT * 10:
            continue
        
        bar_spread = spread_col[bar_pos] if bar_pos < len(spread_col) else SPREAD_POINTS * POINT
        
        sl_dist = atr * sl_atr_mult
        tp_dist = sl_dist * rr_ratio
        
        # Apply slippage to entry
        if signal == 1:
            entry_price += slippage_price  # buy at worse price
            sl_price = entry_price - sl_dist
            tp_price = entry_price + tp_dist
        else:
            entry_price -= slippage_price  # sell at worse price
            sl_price = entry_price + sl_dist
            tp_price = entry_price - tp_dist
        
        result = None
        exit_price = None
        exit_bar = None
        
        for j in range(1, max_bars + 1):
            future_pos = bar_pos + j
            if future_pos >= len(df_full):
                break
            
            h = df_full['high'].iloc[future_pos]
            l = df_full['low'].iloc[future_pos]
            
            if signal == 1:
                if l <= sl_price:
                    result, exit_price, exit_bar = 'SL', sl_price, j
                    break
                elif h >= tp_price:
                    result, exit_price, exit_bar = 'TP', tp_price, j
                    break
            else:
                if h >= sl_price:
                    result, exit_price, exit_bar = 'SL', sl_price, j
                    break
                elif l <= tp_price:
                    result, exit_price, exit_bar = 'TP', tp_price, j
                    break
        
        if result is None:
            future_pos = min(bar_pos + max_bars, len(df_full) - 1)
            exit_price = df_full['close'].iloc[future_pos]
            exit_bar = max_bars
            result = 'TIMEOUT'
        
        # Apply slippage to exit too
        if signal == 1:
            exit_price -= slippage_price  # close buy at worse price
        else:
            exit_price += slippage_price  # close sell at worse price
        
        pnl_price = (exit_price - entry_price) if signal == 1 else (entry_price - exit_price)
        pnl_price -= bar_spread
        pnl_usd = pnl_price * pnl_per_point * lot_size
        
        equity += pnl_usd
        if equity > peak_equity:
            peak_equity = equity
        dd = (peak_equity - equity) / peak_equity * 100
        if dd > max_dd:
            max_dd = dd
        
        trades.append({
            'bar_idx': bar_pos,
            'signal': 'BUY' if signal == 1 else 'SELL',
            'confidence': max_prob,
            'entry': entry_price,
            'exit': exit_price,
            'result': result,
            'pnl_usd': pnl_usd,
            'equity': equity,
            'bars_held': exit_bar,
            'atr': atr,
            'spread': bar_spread,
        })
    
    if not trades:
        return None
    
    trades_df = pd.DataFrame(trades)
    total_trades = len(trades_df)
    wins = trades_df[trades_df['pnl_usd'] > 0]
    losses = trades_df[trades_df['pnl_usd'] <= 0]
    win_rate = len(wins) / total_trades * 100
    tp_count = len(trades_df[trades_df['result'] == 'TP'])
    sl_count = len(trades_df[trades_df['result'] == 'SL'])
    timeout_count = len(trades_df[trades_df['result'] == 'TIMEOUT'])
    total_pnl = trades_df['pnl_usd'].sum()
    avg_win = wins['pnl_usd'].mean() if len(wins) > 0 else 0
    avg_loss = abs(losses['pnl_usd'].mean()) if len(losses) > 0 else 0
    profit_factor = wins['pnl_usd'].sum() / abs(losses['pnl_usd'].sum()) if len(losses) > 0 and losses['pnl_usd'].sum() != 0 else 0
    avg_rr = avg_win / avg_loss if avg_loss > 0 else 0
    
    buy_trades = trades_df[trades_df['signal'] == 'BUY']
    sell_trades = trades_df[trades_df['signal'] == 'SELL']
    buy_wr = len(buy_trades[buy_trades['pnl_usd']>0])/max(1,len(buy_trades))*100
    sell_wr = len(sell_trades[sell_trades['pnl_usd']>0])/max(1,len(sell_trades))*100
    
    return {
        'trades_df': trades_df, 'total_pnl': total_pnl,
        'win_rate': win_rate, 'max_dd': max_dd,
        'profit_factor': profit_factor, 'equity': equity,
        'avg_rr': avg_rr, 'total_trades': total_trades,
        'tp_count': tp_count, 'sl_count': sl_count,
        'timeout_count': timeout_count,
        'buy_wr': buy_wr, 'sell_wr': sell_wr,
        'buy_count': len(buy_trades), 'sell_count': len(sell_trades),
        'avg_win': avg_win, 'avg_loss': avg_loss,
        'avg_spread': trades_df['spread'].mean(),
        'skipped_hold': skipped_hold,
    }

# ============================================================
# GRID SEARCH: find best R:R + SL + Confidence combination
# ============================================================
print("=" * 70)
print(f"GRID SEARCH for {SYMBOL}: Finding optimal parameters (3-class, with slippage)")
print(f"Spread: {SPREAD_POINTS} pts | Slippage: 2 pts | Tick Value: ${TICK_VALUE} | Point: {POINT}")
print("=" * 70)

configs = []
for rr in [1.3, 1.5, 1.8, 2.0]:
    for sl_mult in [1.5, 2.0, 2.5]:
        for conf in [0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
            configs.append((rr, sl_mult, conf))

print(f"Testing {len(configs)} configurations...\n")

all_results = []
for rr, sl_mult, conf in configs:
    r = backtest_xgb(model, feature_df_full, df, scaler, feature_cols,
                    rr_ratio=rr, sl_atr_mult=sl_mult,
                    confidence_threshold=conf, lot_size=0.01, max_bars=48,
                    slippage_points=2)
    if r and r['total_trades'] >= 50:
        r['rr_ratio'] = rr
        r['sl_mult'] = sl_mult
        r['conf_thresh'] = conf
        all_results.append(r)

# Sort by composite score
for r in all_results:
    wr_bonus = 2.0 if r['win_rate'] >= 40 else (1.0 if r['win_rate'] >= 37 else 0.5)
    pf_bonus = r['profit_factor'] if r['profit_factor'] > 0 else 0.01
    r['score'] = pf_bonus * np.sqrt(r['total_trades']) * wr_bonus

all_results.sort(key=lambda x: x['score'], reverse=True)

print(f"\n{'='*95}")
print(f"{'RR':>4} {'SL':>5} {'Conf':>5} | {'Trades':>6} {'WR%':>5} {'PF':>5} {'PnL':>10} {'DD%':>5} {'R:R':>5} {'Hold':>5} | Score")
print(f"{'='*95}")
for r in all_results[:15]:
    wr_flag = "✓" if r['win_rate'] >= 40 else " "
    print(f"{r['rr_ratio']:>4.1f} {r['sl_mult']:>5.1f} {r['conf_thresh']:>5.2f} | "
          f"{r['total_trades']:>6} {r['win_rate']:>5.1f}{wr_flag} {r['profit_factor']:>5.2f} "
          f"${r['total_pnl']:>9.2f} {r['max_dd']:>5.1f} {r['avg_rr']:>5.2f} {r.get('skipped_hold',0):>5} | {r['score']:.1f}")

# Select best with WR >= 40%
wr40_results = [r for r in all_results if r['win_rate'] >= 40]
if wr40_results:
    best = wr40_results[0]
    print(f"\n{'='*70}")
    print(f"BEST (WR>=40%): RR={best['rr_ratio']}, SL={best['sl_mult']}, Conf={best['conf_thresh']:.2f}")
    print(f"Trades: {best['total_trades']} | WR: {best['win_rate']:.1f}% | PF: {best['profit_factor']:.2f}")
    print(f"PnL: ${best['total_pnl']:.2f} | DD: {best['max_dd']:.1f}% | R:R: {best['avg_rr']:.2f}")
    print(f"BUY: {best['buy_count']}({best['buy_wr']:.0f}%) SELL: {best['sell_count']}({best['sell_wr']:.0f}%)")
    print(f"Skipped HOLD: {best.get('skipped_hold', 0)} bars")
    print(f"TP/SL/Timeout: {best['tp_count']}/{best['sl_count']}/{best['timeout_count']}")
    print(f"Avg Spread: {best['avg_spread']:.5f} price")
    print(f"{'='*70}")
    results = best
else:
    best = all_results[0] if all_results else None
    if best:
        print(f"\nNo config with WR>=40%. Best overall:")
        print(f"RR={best['rr_ratio']}, SL={best['sl_mult']}, Conf={best['conf_thresh']:.2f}")
        print(f"WR: {best['win_rate']:.1f}% | PF: {best['profit_factor']:.2f} | PnL: ${best['total_pnl']:.2f}")
        results = best
    else:
        print("\nNo valid results found.")
        results = None

In [ ]:
# ============================================================
# BACKTEST VISUALIZATION
# ============================================================

if results is not None:
    trades_df = results['trades_df']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # 1. Equity Curve
    ax = axes[0, 0]
    ax.plot(trades_df['equity'].values, linewidth=1.2, color='green')
    ax.axhline(y=10000, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
    ax.set_title(f"Equity Curve (PF={results['profit_factor']:.2f}, WR={results['win_rate']:.1f}%)")
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Equity ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. Cumulative PnL
    ax = axes[0, 1]
    cum_pnl = trades_df['pnl_usd'].cumsum()
    ax.plot(cum_pnl.values, linewidth=1.2, color='blue')
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"Cumulative PnL (Total: ${results['total_pnl']:.2f})")
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative PnL ($)')
    ax.grid(True, alpha=0.3)
    
    # 3. Trade PnL Distribution
    ax = axes[1, 0]
    wins = trades_df[trades_df['pnl_usd'] > 0]['pnl_usd']
    losses = trades_df[trades_df['pnl_usd'] <= 0]['pnl_usd']
    ax.hist(wins, bins=50, alpha=0.6, color='green', label=f'Wins ({len(wins)})')
    ax.hist(losses, bins=50, alpha=0.6, color='red', label=f'Losses ({len(losses)})')
    ax.axvline(x=0, color='black', linewidth=1)
    ax.set_title('Trade PnL Distribution')
    ax.set_xlabel('PnL ($)')
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 4. Drawdown
    ax = axes[1, 1]
    equity_arr = trades_df['equity'].values
    peak = np.maximum.accumulate(equity_arr)
    dd_pct = (peak - equity_arr) / peak * 100
    ax.fill_between(range(len(dd_pct)), dd_pct, color='red', alpha=0.3)
    ax.set_title(f"Drawdown (Max: {results['max_dd']:.1f}%)")
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Drawdown (%)')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('backtest_v4_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Monthly breakdown
    print("\n📊 Summary Stats:")
    print(f"  Starting Capital: $10,000")
    print(f"  Final Equity:     ${results['equity']:.2f}")
    print(f"  Return:           {(results['equity']-10000)/10000*100:.1f}%")
    print(f"  Profit Factor:    {results['profit_factor']:.2f}")
    print(f"  Win Rate:         {results['win_rate']:.1f}%")
    print(f"  Avg R:R:          {results['avg_rr']:.2f}")
    print(f"  Max Drawdown:     {results['max_dd']:.1f}%")
    print(f"  Total Trades:     {len(trades_df)}")
    print(f"  Conf Threshold:   {results.get('conf_thresh', 'N/A')}")
else:
    print("No profitable backtest results to visualize.")

## STEP 7a: Backtest M15 Signal + M1 Execution (Simulasi MT5 Strategy Tester)
Mensimulasikan backtest seperti **MT5 Strategy Tester**:
- **Sinyal dihasilkan dari M15** — model predict pada setiap bar M15 (menggunakan model yang sudah di-train)
- **Eksekusi dicek per M1 bar** — SL/TP dievaluasi setiap 1 menit, bukan per 15 menit
- Ini **jauh lebih realistis** karena harga bisa hit SL/TP di antara bar M15
- Download M1 data sebanyak `NUM_BARS_M1` (default: 50000 = ~35 hari trading)
- M15 signal dari bar yang overlap dengan periode M1

In [ ]:
# ============================================================
# BACKTEST: M15 Signal + M1 Execution (MT5 Strategy Tester Style)
# ============================================================
# >>> ATUR JUMLAH BAR M1 DI SINI <<<
NUM_BARS_M1 = 95000   # Jumlah bar M1 untuk simulasi eksekusi
# ============================================================

print("=" * 70)
print(f"MT5-STYLE BACKTEST: M15 Signal + M1 Execution - {SYMBOL}")
print(f"M1 bars: {NUM_BARS_M1} | Model: XGBoost (trained on M15)")
print("=" * 70)

# --- 1. Download M1 data ---
print("\n[1/4] Downloading M1 data...")
rates_m1 = mt5.copy_rates_from_pos(SYMBOL, mt5.TIMEFRAME_M1, 0, NUM_BARS_M1)
if rates_m1 is None or len(rates_m1) == 0:
    raise RuntimeError(f"FAILED to download M1 data: {mt5.last_error()}")

df_m1 = pd.DataFrame(rates_m1)
df_m1['time'] = pd.to_datetime(df_m1['time'], unit='s')
df_m1.set_index('time', inplace=True)
df_m1.rename(columns={'tick_volume': 'volume'}, inplace=True)
print(f"  M1 data: {len(df_m1)} bars")
print(f"  Period : {df_m1.index[0]} to {df_m1.index[-1]}")

# --- 2. Download M15 data covering the same period as M1 ---
print("\n[2/4] Downloading M15 data (same period as M1)...")
m15_bars_needed = NUM_BARS_M1 // 15 + 2000  # extra for feature warmup
rates_m15 = mt5.copy_rates_from_pos(SYMBOL, mt5.TIMEFRAME_M15, 0, m15_bars_needed)
if rates_m15 is None or len(rates_m15) == 0:
    raise RuntimeError(f"FAILED to download M15 data: {mt5.last_error()}")

df_m15_bt = pd.DataFrame(rates_m15)
df_m15_bt['time'] = pd.to_datetime(df_m15_bt['time'], unit='s')
df_m15_bt.set_index('time', inplace=True)
df_m15_bt.rename(columns={'tick_volume': 'volume'}, inplace=True)
print(f"  M15 data: {len(df_m15_bt)} bars")
print(f"  Period : {df_m15_bt.index[0]} to {df_m15_bt.index[-1]}")

# Download H1 for multi-timeframe context
h1_bars_needed = m15_bars_needed // 4 + 500
rates_h1_bt = mt5.copy_rates_from_pos(SYMBOL, mt5.TIMEFRAME_H1, 0, h1_bars_needed)
df_h1_bt = None
if rates_h1_bt is not None and len(rates_h1_bt) > 0:
    df_h1_bt = pd.DataFrame(rates_h1_bt)
    df_h1_bt['time'] = pd.to_datetime(df_h1_bt['time'], unit='s')
    df_h1_bt.set_index('time', inplace=True)
    df_h1_bt.rename(columns={'tick_volume': 'volume'}, inplace=True)
    print(f"  H1 data: {len(df_h1_bt)} bars")

# --- 3. Compute features on M15 data & generate signals ---
print("\n[3/4] Computing M15 features & generating signals...")

# ATR for M15
tr_m15 = np.maximum(
    df_m15_bt['high'].values - df_m15_bt['low'].values,
    np.maximum(
        np.abs(df_m15_bt['high'].values - np.roll(df_m15_bt['close'].values, 1)),
        np.abs(df_m15_bt['low'].values - np.roll(df_m15_bt['close'].values, 1))
    )
)
tr_m15[0] = df_m15_bt['high'].values[0] - df_m15_bt['low'].values[0]
df_m15_bt['atr'] = pd.Series(tr_m15).rolling(14).mean().values

# Full feature pipeline on M15 (same as training)
print("  S/R features...")
sh_m15, sl_m15 = find_swing_points(df_m15_bt, left=5, right=5)
sr_f = compute_sr_features(df_m15_bt, sh_m15, sl_m15)
print("  Trendline features...")
tl_f = compute_trendline_features(df_m15_bt, sh_m15, sl_m15)
print("  Market Structure...")
ms_f = compute_market_structure(df_m15_bt)
print("  Technical + ADX + Volume...")
ti_f = compute_technical_indicators(df_m15_bt)
print("  SMC features...")
smc_f = compute_smc_features(df_m15_bt, sh_m15, sl_m15)

all_f = {}
all_f.update(sr_f)
all_f.update(tl_f)
all_f.update(ms_f)
all_f.update(ti_f)
all_f['candle_body_pct'] = (df_m15_bt['close'].values - df_m15_bt['open'].values) / (df_m15_bt['atr'].values + 1e-10)
all_f['candle_range_pct'] = (df_m15_bt['high'].values - df_m15_bt['low'].values) / (df_m15_bt['atr'].values + 1e-10)
all_f.update(smc_f)

# Time features
hours_bt = df_m15_bt.index.hour.values
dow_bt = df_m15_bt.index.dayofweek.values
all_f['hour_sin'] = np.sin(2 * np.pi * hours_bt / 24.0)
all_f['hour_cos'] = np.cos(2 * np.pi * hours_bt / 24.0)
all_f['day_sin'] = np.sin(2 * np.pi * dow_bt / 5.0)
all_f['day_cos'] = np.cos(2 * np.pi * dow_bt / 5.0)
all_f['session_asian'] = ((hours_bt >= 0) & (hours_bt < 8)).astype(float)
all_f['session_london'] = ((hours_bt >= 7) & (hours_bt < 16)).astype(float)
all_f['session_newyork'] = ((hours_bt >= 13) & (hours_bt < 22)).astype(float)
all_f['session_overlap'] = ((hours_bt >= 13) & (hours_bt < 16)).astype(float)
all_f['is_high_activity'] = (((hours_bt >= 8) & (hours_bt < 11)) | ((hours_bt >= 13) & (hours_bt < 16))).astype(float)

# H1 context
print("  H1 multi-timeframe context...")
h1_f = compute_h1_context_features(df_m15_bt, df_h1_bt)
all_f.update(h1_f)

feat_df_m15 = pd.DataFrame(all_f, index=df_m15_bt.index)
feat_df_m15 = feat_df_m15.iloc[60:].copy()
for col in feat_df_m15.columns:
    feat_df_m15[col] = feat_df_m15[col].clip(-10, 10)
feat_df_m15 = feat_df_m15.dropna()

# Build lag features (same as training)
df_bt_sig = feat_df_m15.copy()
lag_cols = [
    'dist_nearest_support', 'dist_nearest_resistance', 'rsi',
    'macd_normalized', 'return_1', 'trend_score', 'bb_position',
    'price_position_in_sr', 'candle_body_pct',
    'smc_net_score', 'structure_trend', 'premium_discount',
    'bullish_ob_dist', 'bearish_ob_dist',
]
for lag in [1, 2, 3, 5]:
    for col in lag_cols:
        if col in df_bt_sig.columns:
            df_bt_sig[f'{col}_lag{lag}'] = df_bt_sig[col].shift(lag)

change_cols = [
    'dist_nearest_support', 'rsi', 'macd_normalized',
    'price_position_in_sr', 'smc_net_score', 'premium_discount',
]
for col in change_cols:
    if col in df_bt_sig.columns:
        df_bt_sig[f'{col}_change'] = df_bt_sig[col].diff()

df_bt_sig = df_bt_sig.dropna()

# Ensure feature columns match
missing_cols = [c for c in feature_cols if c not in df_bt_sig.columns]
if missing_cols:
    print(f"  WARNING: Missing {len(missing_cols)} features, filling 0")
    for c in missing_cols:
        df_bt_sig[c] = 0.0

X_sig = df_bt_sig[feature_cols].values.astype(np.float32)
X_sig = np.clip(X_sig, -10, 10)
X_sig_scaled = scaler.transform(X_sig)
probs_m15 = model.predict_proba(X_sig_scaled)
m15_valid_idx = df_bt_sig.index

print(f"  M15 signals: {len(probs_m15)} bars with predictions")

# --- 4. Backtest: M15 signal → M1 execution ---
print("\n[4/4] Running MT5-style backtest (M15 signal → M1 execution)...")

# Build M1 lookup: timestamp → M1 position
m1_times = df_m1.index.values
m1_high = df_m1['high'].values
m1_low = df_m1['low'].values
m1_close = df_m1['close'].values
m1_spread = df_m1['spread'].values * POINT if 'spread' in df_m1.columns else np.full(len(df_m1), SPREAD_POINTS * POINT)

# M15 bar timestamp → position in M15 raw data for ATR lookup
m15_ts_to_pos = {ts: pos for pos, ts in enumerate(df_m15_bt.index)}

pnl_per_point = TICK_VALUE / TICK_SIZE
n_m1 = len(df_m1)
m1_start_time = df_m1.index[0]
m1_end_time = df_m1.index[-1]

def backtest_m15_signal_m1_exec(probs, valid_idx, df_m15_raw, df_m1_raw,
                                 m1_times_arr, m1_high_arr, m1_low_arr, m1_close_arr, m1_spread_arr,
                                 rr_ratio=2.0, sl_atr_mult=1.5, confidence_threshold=0.55,
                                 lot_size=0.01, max_m1_bars=720, slippage_points=2,
                                 test_ratio=0.30):
    """
    MT5 Strategy Tester style:
    - Signal dari M15 predictions
    - Entry di close M15 bar
    - SL/TP dicek setiap M1 bar (granular execution)
    - max_m1_bars = timeout dalam M1 bars (720 = 48 x M15 bars = 12 jam)
    """
    slippage_price = slippage_points * POINT
    n_signals = len(probs)
    test_start = int(n_signals * (1 - test_ratio))

    trades = []
    equity = 10000.0
    peak_equity = equity
    max_dd = 0
    skipped_hold = 0
    no_m1_data = 0

    for i in range(test_start, n_signals):
        # --- M15 Signal ---
        prob_sell = probs[i][0]
        prob_buy = probs[i][1]
        prob_hold = probs[i][2] if probs.shape[1] > 2 else 0

        pred_class = np.argmax(probs[i])
        if pred_class == 2:
            skipped_hold += 1
            continue

        max_prob = max(prob_sell, prob_buy)
        if max_prob < confidence_threshold:
            continue

        signal = 1 if prob_buy > prob_sell else 0

        # M15 bar info
        m15_ts = valid_idx[i]
        m15_pos = m15_ts_to_pos.get(m15_ts, None)
        if m15_pos is None:
            continue

        entry_price = df_m15_raw['close'].iloc[m15_pos]
        atr_val = df_m15_raw['atr'].iloc[m15_pos]
        if atr_val is None or np.isnan(atr_val) or atr_val < POINT * 10:
            continue

        # Find M1 bar right after M15 bar close
        # M15 bar closes at m15_ts + 15min, so the next M1 bar starts at m15_ts + 15min
        m15_close_time = np.datetime64(m15_ts + pd.Timedelta(minutes=15))
        m1_entry_idx = np.searchsorted(m1_times_arr, m15_close_time, side='left')
        if m1_entry_idx >= n_m1 - 1:
            no_m1_data += 1
            continue

        bar_spread = m1_spread_arr[m1_entry_idx]

        # SL/TP from M15 ATR
        sl_dist = atr_val * sl_atr_mult
        tp_dist = sl_dist * rr_ratio

        # Apply slippage to entry
        if signal == 1:
            entry_price += slippage_price
            sl_price = entry_price - sl_dist
            tp_price = entry_price + tp_dist
        else:
            entry_price -= slippage_price
            sl_price = entry_price + sl_dist
            tp_price = entry_price - tp_dist

        # --- M1 Execution: check SL/TP per M1 bar ---
        result = None
        exit_price = None
        exit_bar = None

        for j in range(1, max_m1_bars + 1):
            m1_pos = m1_entry_idx + j
            if m1_pos >= n_m1:
                break

            h = m1_high_arr[m1_pos]
            l = m1_low_arr[m1_pos]

            if signal == 1:  # BUY
                if l <= sl_price:
                    result, exit_price, exit_bar = 'SL', sl_price, j
                    break
                elif h >= tp_price:
                    result, exit_price, exit_bar = 'TP', tp_price, j
                    break
            else:  # SELL
                if h >= sl_price:
                    result, exit_price, exit_bar = 'SL', sl_price, j
                    break
                elif l <= tp_price:
                    result, exit_price, exit_bar = 'TP', tp_price, j
                    break

        if result is None:
            timeout_pos = min(m1_entry_idx + max_m1_bars, n_m1 - 1)
            exit_price = m1_close_arr[timeout_pos]
            exit_bar = min(max_m1_bars, timeout_pos - m1_entry_idx)
            result = 'TIMEOUT'

        # Apply slippage to exit
        if signal == 1:
            exit_price -= slippage_price
        else:
            exit_price += slippage_price

        pnl_price = (exit_price - entry_price) if signal == 1 else (entry_price - exit_price)
        pnl_price -= bar_spread
        pnl_usd = pnl_price * pnl_per_point * lot_size

        equity += pnl_usd
        if equity > peak_equity:
            peak_equity = equity
        dd = (peak_equity - equity) / peak_equity * 100
        if dd > max_dd:
            max_dd = dd

        trades.append({
            'bar_idx': m1_entry_idx,
            'signal': 'BUY' if signal == 1 else 'SELL',
            'confidence': max_prob,
            'entry': entry_price,
            'exit': exit_price,
            'result': result,
            'pnl_usd': pnl_usd,
            'equity': equity,
            'bars_held_m1': exit_bar,
            'bars_held_m15': exit_bar / 15.0,
            'atr': atr_val,
            'spread': bar_spread,
            'entry_time': m15_ts,
        })

    if not trades:
        return None

    trades_df = pd.DataFrame(trades)
    total_trades = len(trades_df)
    wins = trades_df[trades_df['pnl_usd'] > 0]
    losses = trades_df[trades_df['pnl_usd'] <= 0]
    win_rate = len(wins) / total_trades * 100
    tp_count = len(trades_df[trades_df['result'] == 'TP'])
    sl_count = len(trades_df[trades_df['result'] == 'SL'])
    timeout_count = len(trades_df[trades_df['result'] == 'TIMEOUT'])
    total_pnl = trades_df['pnl_usd'].sum()
    avg_win = wins['pnl_usd'].mean() if len(wins) > 0 else 0
    avg_loss = abs(losses['pnl_usd'].mean()) if len(losses) > 0 else 0
    profit_factor = wins['pnl_usd'].sum() / abs(losses['pnl_usd'].sum()) if len(losses) > 0 and losses['pnl_usd'].sum() != 0 else 0
    avg_rr = avg_win / avg_loss if avg_loss > 0 else 0

    buy_trades = trades_df[trades_df['signal'] == 'BUY']
    sell_trades = trades_df[trades_df['signal'] == 'SELL']
    buy_wr = len(buy_trades[buy_trades['pnl_usd']>0])/max(1,len(buy_trades))*100
    sell_wr = len(sell_trades[sell_trades['pnl_usd']>0])/max(1,len(sell_trades))*100

    return {
        'trades_df': trades_df, 'total_pnl': total_pnl,
        'win_rate': win_rate, 'max_dd': max_dd,
        'profit_factor': profit_factor, 'equity': equity,
        'avg_rr': avg_rr, 'total_trades': total_trades,
        'tp_count': tp_count, 'sl_count': sl_count,
        'timeout_count': timeout_count,
        'buy_wr': buy_wr, 'sell_wr': sell_wr,
        'buy_count': len(buy_trades), 'sell_count': len(sell_trades),
        'avg_win': avg_win, 'avg_loss': avg_loss,
        'avg_spread': trades_df['spread'].mean(),
        'skipped_hold': skipped_hold,
        'no_m1_data': no_m1_data,
        'avg_bars_held_m1': trades_df['bars_held_m1'].mean(),
    }

# ============================================================
# GRID SEARCH on M15-signal + M1-execution
# ============================================================
configs_m1 = []
for rr in [1.3, 1.5, 1.8, 2.0]:
    for sl_mult in [1.5, 2.0, 2.5]:
        for conf in [0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
            configs_m1.append((rr, sl_mult, conf))

print(f"  Testing {len(configs_m1)} configurations...\n")

all_results_m1 = []
for rr, sl_mult, conf in configs_m1:
    r = backtest_m15_signal_m1_exec(
        probs_m15, m15_valid_idx, df_m15_bt, df_m1,
        m1_times, m1_high, m1_low, m1_close, m1_spread,
        rr_ratio=rr, sl_atr_mult=sl_mult,
        confidence_threshold=conf, lot_size=0.01,
        max_m1_bars=720,  # 720 M1 bars = 48 M15 bars = 12 jam
        slippage_points=2, test_ratio=0.30
    )
    if r and r['total_trades'] >= 20:
        r['rr_ratio'] = rr
        r['sl_mult'] = sl_mult
        r['conf_thresh'] = conf
        all_results_m1.append(r)

# Score & sort
for r in all_results_m1:
    wr_bonus = 2.0 if r['win_rate'] >= 40 else (1.0 if r['win_rate'] >= 37 else 0.5)
    pf_bonus = r['profit_factor'] if r['profit_factor'] > 0 else 0.01
    r['score'] = pf_bonus * np.sqrt(r['total_trades']) * wr_bonus

all_results_m1.sort(key=lambda x: x['score'], reverse=True)

print(f"\n{'='*100}")
print(f"MT5-STYLE BACKTEST: M15 Signal + M1 Execution ({NUM_BARS_M1} M1 bars) - {SYMBOL}")
print(f"{'='*100}")
print(f"{'RR':>4} {'SL':>5} {'Conf':>5} | {'Trades':>6} {'WR%':>5} {'PF':>5} {'PnL':>10} {'DD%':>5} {'R:R':>5} {'Hold':>5} {'AvgM1':>6} | Score")
print(f"{'='*100}")
for r in all_results_m1[:15]:
    wr_flag = "✓" if r['win_rate'] >= 40 else " "
    print(f"{r['rr_ratio']:>4.1f} {r['sl_mult']:>5.1f} {r['conf_thresh']:>5.2f} | "
          f"{r['total_trades']:>6} {r['win_rate']:>5.1f}{wr_flag} {r['profit_factor']:>5.2f} "
          f"${r['total_pnl']:>9.2f} {r['max_dd']:>5.1f} {r['avg_rr']:>5.2f} "
          f"{r.get('skipped_hold',0):>5} {r['avg_bars_held_m1']:>6.0f} | {r['score']:.1f}")

# Select best
wr40_m1 = [r for r in all_results_m1 if r['win_rate'] >= 40]
if wr40_m1:
    best_m1 = wr40_m1[0]
elif all_results_m1:
    best_m1 = all_results_m1[0]
else:
    best_m1 = None

if best_m1:
    print(f"\n{'='*70}")
    print(f"BEST MT5-STYLE: RR={best_m1['rr_ratio']}, SL={best_m1['sl_mult']}, Conf={best_m1['conf_thresh']:.2f}")
    print(f"Trades: {best_m1['total_trades']} | WR: {best_m1['win_rate']:.1f}% | PF: {best_m1['profit_factor']:.2f}")
    print(f"PnL: ${best_m1['total_pnl']:.2f} | DD: {best_m1['max_dd']:.1f}% | R:R: {best_m1['avg_rr']:.2f}")
    print(f"BUY: {best_m1['buy_count']}({best_m1['buy_wr']:.0f}%) SELL: {best_m1['sell_count']}({best_m1['sell_wr']:.0f}%)")
    print(f"Skipped HOLD: {best_m1.get('skipped_hold', 0)} | No M1 data: {best_m1.get('no_m1_data', 0)}")
    print(f"TP/SL/Timeout: {best_m1['tp_count']}/{best_m1['sl_count']}/{best_m1['timeout_count']}")
    print(f"Avg Hold: {best_m1['avg_bars_held_m1']:.0f} M1 bars ({best_m1['avg_bars_held_m1']/15:.1f} M15 bars)")
    print(f"Avg Spread: {best_m1['avg_spread']:.5f} price")
    print(f"{'='*70}")
    results_m1 = best_m1
else:
    print("\nNo valid results found.")
    results_m1 = None

# --- Visualization ---
if best_m1 is not None:
    trades_df_m1 = best_m1['trades_df']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'MT5-Style Backtest: M15 Signal + M1 Execution - {SYMBOL} ({NUM_BARS_M1} M1 bars)',
                 fontsize=13, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(trades_df_m1['equity'].values, linewidth=1.2, color='green')
    ax.axhline(y=10000, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
    ax.set_title(f"Equity Curve (PF={best_m1['profit_factor']:.2f}, WR={best_m1['win_rate']:.1f}%)")
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Equity ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    cum_pnl = trades_df_m1['pnl_usd'].cumsum()
    ax.plot(cum_pnl.values, linewidth=1.2, color='blue')
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"Cumulative PnL (Total: ${best_m1['total_pnl']:.2f})")
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative PnL ($)')
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    wins_m1 = trades_df_m1[trades_df_m1['pnl_usd'] > 0]['pnl_usd']
    losses_m1 = trades_df_m1[trades_df_m1['pnl_usd'] <= 0]['pnl_usd']
    ax.hist(wins_m1, bins=50, alpha=0.6, color='green', label=f'Wins ({len(wins_m1)})')
    ax.hist(losses_m1, bins=50, alpha=0.6, color='red', label=f'Losses ({len(losses_m1)})')
    ax.axvline(x=0, color='black', linewidth=1)
    ax.set_title('Trade PnL Distribution')
    ax.set_xlabel('PnL ($)')
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    equity_arr_m1 = trades_df_m1['equity'].values
    peak_m1 = np.maximum.accumulate(equity_arr_m1)
    dd_pct_m1 = (peak_m1 - equity_arr_m1) / peak_m1 * 100
    ax.fill_between(range(len(dd_pct_m1)), dd_pct_m1, color='red', alpha=0.3)
    ax.set_title(f"Drawdown (Max: {best_m1['max_dd']:.1f}%)")
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Drawdown (%)')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('backtest_m15sig_m1exec_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n📊 MT5-Style Backtest Summary:")
    print(f"  Signal TF:        M15 (model prediction)")
    print(f"  Execution TF:     M1 ({NUM_BARS_M1} bars)")
    print(f"  Starting Capital: $10,000")
    print(f"  Final Equity:     ${best_m1['equity']:.2f}")
    print(f"  Return:           {(best_m1['equity']-10000)/10000*100:.1f}%")
    print(f"  Profit Factor:    {best_m1['profit_factor']:.2f}")
    print(f"  Win Rate:         {best_m1['win_rate']:.1f}%")
    print(f"  Avg R:R:          {best_m1['avg_rr']:.2f}")
    print(f"  Max Drawdown:     {best_m1['max_dd']:.1f}%")
    print(f"  Avg Hold Time:    {best_m1['avg_bars_held_m1']:.0f} min ({best_m1['avg_bars_held_m1']/60:.1f} hours)")
    print(f"  Total Trades:     {len(trades_df_m1)}")

## STEP 7b: Walk-Forward Validation
Menguji apakah model **robust di semua periode**, bukan hanya bulan terakhir.
- Data dibagi menjadi beberapa window (fold)
- Setiap fold: train di window lama, test di window baru
- Model yang bagus = **profitable di mayoritas folds**

In [ ]:
# ============================================================
# WALK-FORWARD VALIDATION - 10 folds + slippage + 3-class
# Anti-Overfitting Version
# ============================================================

def walk_forward_backtest_xgb(df_full, n_folds=10, train_months=6, test_months=1,
                               slippage_points=2,
                          df_h1_data=None):
    """
    Walk-Forward: train XGBoost di window lama, backtest di window baru.
    10 folds, 1-month OOS, with slippage and 3-class model.
    Anti-overfitting params applied.
    """
    from sklearn.preprocessing import RobustScaler
    
    print("="*70)
    print(f"WALK-FORWARD VALIDATION for {SYMBOL} (XGBoost, 3-class + slippage)")
    print("="*70)
    
    total_bars = len(df_full)
    bars_per_day = total_bars / max(1, (df_full.index[-1] - df_full.index[0]).days)
    bars_per_month = int(bars_per_day * 30)
    train_bars = bars_per_month * train_months
    test_bars = bars_per_month * test_months
    
    print(f"Total bars: {total_bars} | ~{bars_per_month} bars/month")
    print(f"Train window: {train_months}mo (~{train_bars} bars)")
    print(f"Test window: {test_months}mo (~{test_bars} bars)")
    print(f"Slippage: {slippage_points} points")
    
    folds = []
    for i in range(n_folds):
        test_end = total_bars - i * test_bars
        test_start = test_end - test_bars
        train_end = test_start
        train_start = max(0, train_end - train_bars)
        
        if train_end - train_start < bars_per_month * 3:
            break
        if test_start < 0 or test_end <= test_start:
            break
            
        folds.append((train_start, train_end, test_start, test_end))
    
    folds.reverse()
    n_folds = len(folds)
    print(f"Valid folds: {n_folds}\n")
    
    all_fold_results = []
    all_fold_trades = []
    
    for fold_idx, (tr_s, tr_e, te_s, te_e) in enumerate(folds):
        df_train = df_full.iloc[tr_s:tr_e].copy()
        df_test = df_full.iloc[te_s:te_e].copy()
        
        print(f"{'='*60}")
        print(f"FOLD {fold_idx+1}/{n_folds}")
        print(f"  Train: {df_train.index[0].date()} → {df_train.index[-1].date()} ({len(df_train)} bars)")
        print(f"  Test:  {df_test.index[0].date()} → {df_test.index[-1].date()} ({len(df_test)} bars)")
        
        try:
            fold_labeled, fold_full = create_dataset_v5(df_train, future_bars=FUTURE_BARS, df_h1_data=df_h1_data)
        except Exception as e:
            print(f"  SKIP: {e}")
            continue
        
        if len(fold_labeled) < 300:
            print(f"  SKIP: too few samples ({len(fold_labeled)})")
            continue
        
        base_cols = [c for c in fold_labeled.columns if c != 'label']
        df_lags = fold_labeled.copy()
        
        lag_cols = [
            'dist_nearest_support', 'dist_nearest_resistance', 'rsi',
            'macd_normalized', 'return_1', 'trend_score', 'bb_position',
            'price_position_in_sr', 'candle_body_pct',
            'smc_net_score', 'structure_trend', 'premium_discount',
            'bullish_ob_dist', 'bearish_ob_dist',
        ]
        for lag in [1, 2, 3, 5]:
            for col in lag_cols:
                if col in df_lags.columns:
                    df_lags[f'{col}_lag{lag}'] = df_lags[col].shift(lag)
        
        change_cols = [
            'dist_nearest_support', 'rsi', 'macd_normalized',
            'price_position_in_sr', 'smc_net_score', 'premium_discount',
        ]
        for col in change_cols:
            if col in df_lags.columns:
                df_lags[f'{col}_change'] = df_lags[col].diff()
        
        df_lags = df_lags.dropna()
        fold_features = [c for c in df_lags.columns if c != 'label']
        
        X = df_lags[fold_features].values.astype(np.float32)
        y = df_lags['label'].values.astype(np.int64)
        X = np.clip(X, -10, 10)
        
        split = int(len(X) * 0.85)
        X_tr, y_tr = X[:split], y[:split]
        X_vl, y_vl = X[split:], y[split:]
        
        fold_scaler = RobustScaler()
        X_tr_s = fold_scaler.fit_transform(X_tr)
        X_vl_s = fold_scaler.transform(X_vl)
        
        if len(X_tr) < 200 or len(X_vl) < 50:
            print(f"  SKIP: insufficient data after split")
            continue
        
        # Train XGBoost model for this fold - ANTI-OVERFITTING params
        num_classes_fold = len(np.unique(y_tr))
        class_counts_fold = np.bincount(y_tr, minlength=num_classes_fold).astype(float)
        class_weights_fold = len(y_tr) / (num_classes_fold * class_counts_fold + 1e-6)
        sample_weights_fold = np.array([class_weights_fold[lb] for lb in y_tr])
        
        fold_model = xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=num_classes_fold,
            eval_metric='mlogloss',
            max_depth=4,                  # reduced from 8
            learning_rate=0.03,           # reduced from 0.05
            n_estimators=1500,            # with early stopping
            subsample=0.6,               # reduced from 0.8
            colsample_bytree=0.6,        # reduced from 0.8
            colsample_bylevel=0.5,       # reduced from 0.7
            colsample_bynode=0.8,        # NEW
            min_child_weight=20,         # increased from 5
            gamma=1.0,                   # increased from 0.1
            reg_alpha=2.0,               # increased from 0.5
            reg_lambda=5.0,              # increased from 2.0
            max_delta_step=1,
            max_leaves=31,               # NEW: limit leaf complexity
            early_stopping_rounds=50,    # Stop when val loss stops improving
            tree_method='hist',
            random_state=42,
            verbosity=0,
        )
        
        fold_model.fit(
            X_tr_s, y_tr,
            sample_weight=sample_weights_fold,
            eval_set=[(X_vl_s, y_vl)],
            verbose=False
        )
        
        val_preds_fold = fold_model.predict(X_vl_s)
        best_val = np.mean(val_preds_fold == y_vl)
        best_iter = fold_model.best_iteration if hasattr(fold_model, 'best_iteration') and fold_model.best_iteration else '?'
        print(f"  Training done: Best Val Acc = {best_val*100:.1f}% (iter {best_iter})")
        
        try:
            test_labeled, test_full = create_dataset_v5(df_test, future_bars=FUTURE_BARS, df_h1_data=df_h1_data)
        except Exception as e:
            print(f"  SKIP test: {e}")
            continue
        
        try:
            fold_result = backtest_on_window_xgb(
                fold_model, test_full, df_test, fold_scaler, fold_features,
                rr_ratio=2.0, sl_atr_mult=2.5, confidence_threshold=0.55,
                lot_size=0.01, max_bars=48, slippage_points=slippage_points
            )
        except Exception as e:
            print(f"  Backtest error: {e}")
            fold_result = None
        
        if fold_result and fold_result['total_trades'] > 0:
            status = "✓ PROFIT" if fold_result['total_pnl'] > 0 else "✗ LOSS"
            print(f"  OOS: WR={fold_result['win_rate']:.1f}% PF={fold_result['profit_factor']:.2f} "
                  f"Trades={fold_result['total_trades']} PnL=${fold_result['total_pnl']:.2f} "
                  f"Hold={fold_result.get('skipped_hold',0)} {status}")
            
            fold_result['fold'] = fold_idx + 1
            fold_result['train_period'] = f"{df_train.index[0].date()} → {df_train.index[-1].date()}"
            fold_result['test_period'] = f"{df_test.index[0].date()} → {df_test.index[-1].date()}"
            all_fold_results.append(fold_result)
            
            if 'trades_df' in fold_result:
                tdf = fold_result['trades_df'].copy()
                tdf['fold'] = fold_idx + 1
                all_fold_trades.append(tdf)
        else:
            print(f"  No trades in test window")
    
    # ============================================================
    # SUMMARY
    # ============================================================
    print(f"\n{'='*70}")
    print("WALK-FORWARD SUMMARY (XGBoost, 3-class + slippage)")
    print(f"{'='*70}")
    
    if not all_fold_results:
        print("  No valid fold results!")
        return [], None
    
    print(f"\n{'Fold':>4} | {'Test Period':<27} | {'Trades':>6} {'WR%':>5} {'PF':>5} {'PnL':>10} {'DD%':>5} {'Hold':>5} | Status")
    print("-"*95)
    
    for r in all_fold_results:
        status = "✓" if r['total_pnl'] > 0 else "✗"
        print(f"  {r['fold']:>2}  | {r['test_period']:<27} | "
              f"{r['total_trades']:>6} {r['win_rate']:>5.1f} {r['profit_factor']:>5.2f} "
              f"${r['total_pnl']:>9.2f} {r['max_dd']:>5.1f} {r.get('skipped_hold',0):>5} | {status}")
    
    profitable_folds = sum(1 for r in all_fold_results if r['total_pnl'] > 0)
    avg_wr = np.mean([r['win_rate'] for r in all_fold_results])
    avg_pf = np.mean([r['profit_factor'] for r in all_fold_results])
    total_pnl = sum(r['total_pnl'] for r in all_fold_results)
    total_trades = sum(r['total_trades'] for r in all_fold_results)
    
    print(f"\n  Profitable Folds: {profitable_folds}/{len(all_fold_results)}")
    print(f"  Avg WR: {avg_wr:.1f}% | Avg PF: {avg_pf:.2f}")
    print(f"  Total OOS Trades: {total_trades}")
    print(f"  Total OOS PnL: ${total_pnl:.2f}")
    
    # Stricter evaluation: need 70%+ profitable folds
    if profitable_folds >= len(all_fold_results) * 0.7:
        print(f"\n  ✓ MODEL ROBUST - {profitable_folds}/{len(all_fold_results)} folds profitable (≥70%)")
        print(f"  → Safe to deploy with periodic retraining")
    elif profitable_folds >= len(all_fold_results) * 0.5:
        print(f"\n  ⚠ MODEL MARGINAL - {profitable_folds}/{len(all_fold_results)} folds profitable")
        print(f"  → Needs improvement, consider more features or different params")
    else:
        print(f"\n  ✗ MODEL NOT ROBUST - {profitable_folds}/{len(all_fold_results)} folds profitable")
        print(f"  → Do NOT deploy, model overfits to recent data")
    
    return all_fold_results, all_fold_trades


def backtest_on_window_xgb(model, feature_df_full, df_full, scaler, feature_cols,
                       rr_ratio=2.0, sl_atr_mult=2.5, confidence_threshold=0.55,
                       lot_size=0.01, max_bars=48, slippage_points=2):
    """Backtest on ENTIRE window (no 70% skip) - for walk-forward OOS testing
    3-class: skips HOLD predictions, adds slippage."""
    
    df_bt = feature_df_full.copy()
    
    lag_cols = [
        'dist_nearest_support', 'dist_nearest_resistance', 'rsi',
        'macd_normalized', 'return_1', 'trend_score', 'bb_position',
        'price_position_in_sr', 'candle_body_pct',
        'smc_net_score', 'structure_trend', 'premium_discount',
        'bullish_ob_dist', 'bearish_ob_dist',
    ]
    for lag in [1, 2, 3, 5]:
        for col in lag_cols:
            if col in df_bt.columns:
                df_bt[f'{col}_lag{lag}'] = df_bt[col].shift(lag)
    
    change_cols = [
        'dist_nearest_support', 'rsi', 'macd_normalized',
        'price_position_in_sr', 'smc_net_score', 'premium_discount',
    ]
    for col in change_cols:
        if col in df_bt.columns:
            df_bt[f'{col}_change'] = df_bt[col].diff()
    
    df_bt = df_bt.dropna()
    
    missing = [c for c in feature_cols if c not in df_bt.columns]
    if missing:
        return None
    
    valid_idx = df_bt.index
    X_all = df_bt[feature_cols].values.astype(np.float32)
    X_all = np.clip(X_all, -10, 10)
    X_scaled = scaler.transform(X_all)
    
    # XGBoost predictions
    all_probs = model.predict_proba(X_scaled)
    
    ts_to_pos = {ts: pos for pos, ts in enumerate(df_full.index)}
    pnl_per_point = TICK_VALUE / TICK_SIZE
    spread_col = df_full['spread'].values * POINT if 'spread' in df_full.columns else np.full(len(df_full), SPREAD_POINTS * POINT)
    slippage_price = slippage_points * POINT
    
    trades = []
    equity = 10000.0
    peak_equity = equity
    max_dd = 0
    skipped_hold = 0
    
    for i in range(len(all_probs) - 1):
        # 3-class: skip HOLD
        pred_class = np.argmax(all_probs[i])
        if pred_class == 2:
            skipped_hold += 1
            continue
        
        prob_sell = all_probs[i][0]
        prob_buy = all_probs[i][1]
        max_prob = max(prob_sell, prob_buy)
        if max_prob < confidence_threshold:
            continue
        
        signal = 1 if prob_buy > prob_sell else 0
        bar_ts = valid_idx[i]
        bar_pos = ts_to_pos.get(bar_ts, None)
        if bar_pos is None or bar_pos >= len(df_full) - 1:
            continue
        
        entry_price = df_full['close'].iloc[bar_pos]
        atr = df_full['atr'].iloc[bar_pos] if 'atr' in df_full.columns else None
        if atr is None or atr < POINT * 10:
            continue
        
        bar_spread = spread_col[bar_pos] if bar_pos < len(spread_col) else SPREAD_POINTS * POINT
        sl_dist = atr * sl_atr_mult
        tp_dist = sl_dist * rr_ratio
        
        # Apply slippage
        if signal == 1:
            entry_price += slippage_price
            sl_price = entry_price - sl_dist
            tp_price = entry_price + tp_dist
        else:
            entry_price -= slippage_price
            sl_price = entry_price + sl_dist
            tp_price = entry_price - tp_dist
        
        result = None
        exit_price = None
        exit_bar = None
        
        for j in range(1, max_bars + 1):
            future_pos = bar_pos + j
            if future_pos >= len(df_full):
                break
            h = df_full['high'].iloc[future_pos]
            l = df_full['low'].iloc[future_pos]
            
            if signal == 1:
                if l <= sl_price:
                    result, exit_price, exit_bar = 'SL', sl_price, j
                    break
                elif h >= tp_price:
                    result, exit_price, exit_bar = 'TP', tp_price, j
                    break
            else:
                if h >= sl_price:
                    result, exit_price, exit_bar = 'SL', sl_price, j
                    break
                elif l <= tp_price:
                    result, exit_price, exit_bar = 'TP', tp_price, j
                    break
        
        if result is None:
            future_pos = min(bar_pos + max_bars, len(df_full) - 1)
            exit_price = df_full['close'].iloc[future_pos]
            exit_bar = max_bars
            result = 'TIMEOUT'
        
        # Exit slippage
        if signal == 1:
            exit_price -= slippage_price
        else:
            exit_price += slippage_price
        
        pnl_price = (exit_price - entry_price) if signal == 1 else (entry_price - exit_price)
        pnl_price -= bar_spread
        pnl_usd = pnl_price * pnl_per_point * lot_size
        
        equity += pnl_usd
        if equity > peak_equity:
            peak_equity = equity
        dd = (peak_equity - equity) / peak_equity * 100
        if dd > max_dd:
            max_dd = dd
        
        trades.append({
            'signal': 'BUY' if signal == 1 else 'SELL',
            'pnl': pnl_usd,
            'result': result,
            'spread': bar_spread,
        })
    
    if not trades:
        return {'total_trades': 0, 'win_rate': 0, 'profit_factor': 0,
                'total_pnl': 0, 'max_dd': 0, 'skipped_hold': skipped_hold}
    
    trades_df = pd.DataFrame(trades)
    wins_mask = trades_df['pnl'] > 0
    gross_profit = trades_df.loc[wins_mask, 'pnl'].sum()
    gross_loss = abs(trades_df.loc[~wins_mask, 'pnl'].sum())
    
    return {
        'total_trades': len(trades_df),
        'win_rate': wins_mask.mean() * 100,
        'profit_factor': gross_profit / max(gross_loss, 0.01),
        'total_pnl': trades_df['pnl'].sum(),
        'max_dd': max_dd,
        'avg_spread': trades_df['spread'].mean(),
        'buy_count': (trades_df['signal']=='BUY').sum(),
        'sell_count': (trades_df['signal']=='SELL').sum(),
        'trades_df': trades_df,
        'skipped_hold': skipped_hold,
    }

# ============================================================
# RUN WALK-FORWARD (10 folds, 1-month OOS each)
# ============================================================
wf_results, wf_trades = walk_forward_backtest_xgb(
    df,
    n_folds=10,          # 10 folds (stricter evaluation)
    train_months=6,      # train on 6 months
    test_months=1,       # test on next 1 month (OOS)
    slippage_points=2,
    df_h1_data=df_h1
)

In [ ]:
# ============================================================
# WALK-FORWARD RESULTS SUMMARY
# ============================================================
print(f"{'='*80}")
print(f"WALK-FORWARD RESULTS for {SYMBOL} (XGBoost)")
print(f"{'='*80}")

if wf_results:
    print(f"\n{'Fold':>4} | {'Test Period':<27} | {'Trades':>6} {'WR%':>6} {'PF':>6} {'PnL':>11} {'DD%':>6} | Status")
    print("-"*90)
    for r in wf_results:
        status = "✓ PROFIT" if r['total_pnl'] > 0 else "✗ LOSS"
        print(f"  {r['fold']:>2}  | {r['test_period']:<27} | "
              f"{r['total_trades']:>6} {r['win_rate']:>6.1f} {r['profit_factor']:>6.2f} "
              f"${r['total_pnl']:>10.2f} {r['max_dd']:>6.1f} | {status}")
    
    profitable = sum(1 for r in wf_results if r['total_pnl'] > 0)
    avg_wr = np.mean([r['win_rate'] for r in wf_results])
    avg_pf = np.mean([r['profit_factor'] for r in wf_results])
    total_pnl = sum(r['total_pnl'] for r in wf_results)
    total_trades = sum(r['total_trades'] for r in wf_results)
    
    print(f"\n  Profitable Folds: {profitable}/{len(wf_results)}")
    print(f"  Avg WR: {avg_wr:.1f}% | Avg PF: {avg_pf:.2f}")
    print(f"  Total OOS Trades: {total_trades}")
    print(f"  Total OOS PnL: ${total_pnl:.2f}")
    
    if profitable >= len(wf_results) * 0.6:
        print(f"\n  ✓ MODEL ROBUST across {profitable}/{len(wf_results)} periods")
    elif profitable >= len(wf_results) * 0.4:
        print(f"\n  ⚠ MODEL MARGINAL - needs improvement")
    else:
        print(f"\n  ✗ MODEL NOT ROBUST - overfits to recent data")
else:
    print("  No walk-forward results available")